In [1]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 17.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (classification_report, accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 하이퍼파라미터 및 경로 설정 (수정됨)
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # [New] 거래 비용 설정 (편도 기준)
    'fee_rate': 0.00025,      # 0.025% 업비트 수수료
    'slippage': 0.00020,      # 0.020% (슬리피지 가정)

    'triple_barrier': {
        'span': 100,              # 변동성 계산 기간
        'pt': 1.0,                # [수정] 익절 배수 (이익 폭 확대: 1.5 -> 2.0)
        'sl': 1.0,                # [수정] 손절 배수 (손실 폭 축소: 1.5 -> 1.0)
        'vertical_barrier': 40  # 최대 보유 기간 (캔들 수)
    },
    'optuna_trials': 5,
    'purge_gap': 10,
    'meta_labeling_threshold': 0.6
}

class FocalLossObjective:
    def __init__(self, alpha, gamma):
        self.alpha = alpha
        self.gamma = gamma

    def get_objective(self, y_true, y_pred):
        # 인자 매핑 (Traceback 기준: 첫 번째 인자가 정답 레이블, 두 번째가 예측값)
        labels = y_true
        preds = y_pred

        # Log-odds -> Probability (Sigmoid)
        preds = 1.0 / (1.0 + np.exp(-preds))
        preds = np.clip(preds, 1e-7, 1.0 - 1e-7)

        pt = np.where(labels == 1, preds, 1 - preds)
        alpha_t = np.where(labels == 1, self.alpha, 1 - self.alpha)

        # Gradient (1차 미분)
        grad = alpha_t * (1 - pt)**self.gamma * (preds - labels)

        # Hessian (2차 미분)
        hess = alpha_t * (1 - pt)**self.gamma * preds * (1 - preds) * \
               (1 + self.gamma * (1 - pt) * np.log(pt))

        return grad, np.maximum(hess, 1e-6)


class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None

        # 데이터셋
        self.X_train = None
        self.X_test = None
        self.y_class_train = None  # 분류 타겟 (0, 1)
        self.y_class_test = None

        # 검증용 수익률 데이터 (학습엔 안 씀)
        self.y_return_test = None

        # 모델
        self.classifier = None

        self.feature_cols = []
        self.best_params_class = {}

    # 1. 데이터 로드
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        # [수정] 더미 데이터 생성 로직 삭제 -> 실패 시 즉시 에러 발생하도록 변경
        if not pd.io.common.file_exists(self.config['file_path']):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {self.config['file_path']}")

        self.df = pd.read_csv(self.config['file_path'])
        self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        self.df = self.df.set_index('datetime').sort_index()
        self.df.dropna(inplace=True)
        print(f"    >>> 데이터 로드 완료: {self.df.shape}")

    # 2. Triple Barrier (Cost-aware Labeling 적용)
    def _get_volatility(self, prices, span=100):
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        """
        수수료와 슬리피지를 고려하여, 비용을 제하고도 수익이 나는 구간만 Label 1로 설정
        """
        print("[2] Triple Barrier 레이블링 (Cost-aware) 적용 중...")

        # 1) 변동성 계산
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        # 2) Triple Barrier 파라미터
        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        # 3) 거래 비용 기반 최소 요구 수익률 계산
        fee_one_way = self.config.get('fee_rate', 0.00025)     # 편도 수수료
        slip_one_way = self.config.get('slippage', 0.00020)    # 편도 슬리피지

        round_trip_cost = 2 * (fee_one_way + slip_one_way)   # ≈ 0.0009

        multiplier   = 1.5        # 비용의 1.5배 정도만 요구
        extra_margin = 0.0005     # 0.05% 추가

        MIN_RET = round_trip_cost * multiplier + extra_margin

        print(f"    >>> round_trip_cost: {round_trip_cost*100:.3f}% "
              f"(fee={fee_one_way*100:.3f}%, slip={slip_one_way*100:.3f}%)")
        print(f"    >>> MIN_RET (success 기준): {MIN_RET*100:.3f}%")

        labels = []   # 0: 실패/손절/미미한 이익, 1: 수수료 제하고도 의미 있는 성공
        returns = []  # 실제 수익률 (검증용)

        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values

        n_samples = len(closes)

        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.001)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = np.nan
            ret = np.nan
            touched = False

            # 4) horizon 내 TP/SL 체크
            for j in range(1, t_final + 1):
                # Take Profit: high >= upper
                if highs[i + j] >= upper:
                    # TP 도달 시 수익률은 목표가(upper) 기준으로 계산 (보수적)
                    ret = (upper - current_price) / current_price

                    # 수수료+슬리피지+마진 포함한 MIN_RET 이상이어야 성공(1)
                    if ret > MIN_RET:
                        label = 1
                    else:
                        label = 0

                    touched = True
                    break

                # Stop Loss: low <= lower
                if lows[i + j] <= lower:
                    # SL 도달 시 수익률은 손절가(lower) 기준
                    ret = (lower - current_price) / current_price
                    label = 0 # 손절은 무조건 실패
                    touched = True
                    break

            # 5) TP/SL 둘 다 안 닿은 경우 → vertical barrier (Time out)
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price

                # timeout 시에도 수익이 MIN_RET 이상이어야 1
                if ret > MIN_RET:
                    label = 1
                else:
                    label = 0

            labels.append(label)
            returns.append(ret)

        # 끝의 남는 구간 NaN 패딩
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns # 검증용

        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # 3. 데이터 분할 및 정규화 (Rolling Scaling)
    def split_and_scale(self):
        print("[3] 데이터 분할 및 스케일링...")
        exclude = ['target_class', 'target_return', 'open', 'high', 'low', 'close', 'volume', 'datetime', 'volatility']
        feature_cols = [c for c in self.df.columns if c not in exclude]

        window_size = 200

        X = self.df[feature_cols]
        r_mean = X.rolling(window_size).mean()
        r_std = X.rolling(window_size).std().replace(0, 1)
        X_scaled = (X - r_mean) / r_std

        valid_idx = X_scaled.dropna().index
        split_idx = int(len(valid_idx) * (1 - self.config['test_size']))

        self.X_train = X_scaled.loc[valid_idx].iloc[:split_idx].clip(-5, 5)
        self.X_test = X_scaled.loc[valid_idx].iloc[split_idx:].clip(-5, 5)

        self.y_class_train = self.df.loc[valid_idx, 'target_class'].iloc[:split_idx]
        self.y_class_test = self.df.loc[valid_idx, 'target_class'].iloc[split_idx:]
        self.y_return_test = self.df.loc[valid_idx, 'target_return'].iloc[split_idx:]

        print(f"    >>> 학습 셋: {self.X_train.shape}")

    # 4. Purged K-Fold CV
    def _purged_cv_score(self, model, X, y):
            kf = KFold(n_splits=3, shuffle=False)
            scores = []
            gap = self.config['purge_gap']

            for tr_idx, val_idx in kf.split(X):
                tr_idx = tr_idx[tr_idx < val_idx[0] - gap]
                if len(tr_idx) < 100: continue

                model.fit(X.iloc[tr_idx], y.iloc[tr_idx], verbose=False)

                # [수정] 기본 임계값(0.5) 기준 예측
                pred = model.predict(X.iloc[val_idx])

                # 평가 지표를 Accuracy로 변경
                scores.append(accuracy_score(y.iloc[val_idx], pred))

            return np.mean(scores) if scores else 0

    # 5. Optuna 튜닝 (Precision 최적화)
    # =============================================================================
    def run_optuna(self):
        # [수정] Target 메시지 변경
        print(f"[4] 하이퍼파라미터 튜닝 (Target: Accuracy, Trials: {self.config['optuna_trials']})")

        def objective_cls(trial):
            # 1. Focal Loss 파라미터 [공격적 조정]
            # alpha: 0.9 이상으로 설정하여 Class 1(상승)을 틀렸을 때의 손실을 극대화
            # 이렇게 해야 모델이 0.5 이상의 확률을 더 자주 배출합니다.
            fl_alpha = trial.suggest_float('fl_alpha', 0.65, 0.85)

            # gamma: 값을 낮춰(0~1) 확률 분포를 0에서 멀어지게 유도 (Standard CE와 유사하게)
            fl_gamma = trial.suggest_float('fl_gamma', 2.5, 4.5)

            # Focal Loss 객체 생성
            fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)

            params = {
                # 트리 구조 (범위 소폭 조정)
                'n_estimators': trial.suggest_int('n_estimators', 400, 800), # 트리 수 증가
                'max_depth': trial.suggest_int('max_depth', 4, 9),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),

                # 샘플링
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),

                # 정규화
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 0.1, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'gamma': trial.suggest_float('xgb_gamma', 0.0, 2.0), # XGBoost gamma도 낮춰서 가지치기 완화

                # 설정
                'objective': fl_obj.get_objective,
                'eval_metric': 'logloss',
                'disable_default_eval_metric': 1,
                'scale_pos_weight': 1.0,

                # 하드웨어
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])

        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")

        return study_cls

    # =============================================================================
    # 6. 최종 모델 학습 및 Meta Labeling 검증
    # =============================================================================
    def train_final_models(self):
        """최종 모델 학습 (Focal Loss 적용)"""
        print("[5] 최종 모델 학습...")

        # 1. Optuna 결과에서 파라미터 분리
        bp = self.best_params_class.copy()
        fl_alpha = bp.pop('fl_alpha')
        fl_gamma = bp.pop('fl_gamma')

        # 이름 충돌 해결 (xgb_gamma -> gamma)
        if 'xgb_gamma' in bp:
            bp['gamma'] = bp.pop('xgb_gamma')

        # 2. Focal Loss 객체 재생성
        fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)

        # 3. 모델 생성
        self.classifier = xgb.XGBClassifier(
            **bp,
            objective=fl_obj.get_objective,
            scale_pos_weight=1.0, # 명시적 초기화
            n_jobs=-1
        )

        self.classifier.fit(self.X_train, self.y_class_train)

        print(f"    >>> 모델 학습 완료 (Alpha={fl_alpha:.2f}, Gamma={fl_gamma:.2f})")
        return self.classifier
        # =========================================================================
    # [Helper] F1-Score 기반 최적 임계값 찾기
    # =========================================================================
    def _find_optimal_threshold(self, pred_probs, y_true):
        """정밀도-재현율 커브를 통해 F1 Score가 최대가 되는 임계값을 반환"""
        precision, recall, thresholds = precision_recall_curve(y_true, pred_probs)

        # F1 Score 계산 (분모 0 방지)
        numerator = 2 * precision * recall
        denominator = precision + recall
        f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)

        best_idx = np.argmax(f1_scores)

        # thresholds 길이는 precision/recall보다 1이 짧음
        optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        max_f1 = f1_scores[best_idx]

        return optimal_threshold, max_f1

    # =========================================================================
    # [Main] 통합 평가 함수 (기존 상세 로직 + 동적 임계값)
    # =========================================================================
    def evaluate(self):
        """
        [통합 평가 로직 - 유연한 임계값 적용]
        1. 0.5 절댓값 제한 해제: Focal Loss의 낮은 확률 분포 특성 반영
        2. 상대적 기준 적용: 전체 예측값 중 상위 25% 이상일 때만 진입 허용 (노이즈 필터링)
        3. F1-Score 최적화: 데이터 기반으로 가장 수익성 높은 임계값 탐색
        """

        # 1. 모델 예측
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]
        probs_series = pd.Series(pred_probs)

        print("\n" + "="*60)
        print(" [ 전략 평가 및 백테스팅 (Flexible Threshold) ]")
        print("="*60)

        # ---------------------------------------------------------
        # Step 1: 비용 설정 및 데이터 진단
        # ---------------------------------------------------------
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00020)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        print(f">>> 적용 왕복 비용: {round_trip_cost*100:.3f}%")
        print("\n[Diagnosis] 예측 확률 분포 요약:")
        print(probs_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

        # ---------------------------------------------------------
        # Step 2: 동적 임계값(Threshold) 산출
        # ---------------------------------------------------------
        opt_th, max_f1 = self._find_optimal_threshold(pred_probs, self.y_class_test)
        print(f"\n[Optimization] F1-Score 최대화 임계값 발견")
        print(f"    >>> Max F1: {max_f1:.4f} at Threshold: {opt_th:.4f}")

        # ---------------------------------------------------------
        # Step 3: 임계값 보정 (Relative Safety Guard)
        # ---------------------------------------------------------
        final_th = opt_th

        # [수정] 0.5 절대 기준 삭제 -> 상대적 기준으로 변경
        # 논리: Focal Loss는 0.5를 못 넘기는 경우가 많음.
        # 하지만 전체 예측값 중 상위 25% 안에는 들어야 "상승 신호"라고 볼 수 있음.
        min_relative_limit = probs_series.quantile(0.75)

        if final_th < min_relative_limit:
            print(f"    >>> [보정] 최적값({final_th:.4f})이 너무 낮아 노이즈 위험이 있습니다.")
            print(f"    >>> 상위 25% 수준인 {min_relative_limit:.4f}로 상향 조정합니다.")
            final_th = min_relative_limit

        # 상한선: 상위 0.1% (너무 극단적인 값) 이상이면 완화 (거래 기회 확보)
        cap_th = probs_series.quantile(0.999)
        if final_th > cap_th:
            print(f"    >>> [보정] 최적값({final_th:.4f})이 지나치게 높습니다. 상위 0.1%({cap_th:.4f})로 조정.")
            final_th = cap_th

        print(f"    >>> 최종 적용 Threshold: {final_th:.4f}")

        # ---------------------------------------------------------
        # Step 4: 백테스팅 (Cooldown & Cost 적용)
        # ---------------------------------------------------------
        returns_arr = self.y_return_test.values
        executed_trades = []

        # 쿨다운: 진입 후 설정된 기간(캔들) 동안은 추가 진입 금지
        cooldown = self.config['triple_barrier']['vertical_barrier']
        next_trade_idx = 0

        print(f"\n[Backtest] 시뮬레이션 시작 (쿨다운: {cooldown} 캔들)...")

        for i in range(len(pred_probs)):
            # 쿨다운 기간 체크
            if i < next_trade_idx:
                continue

            # 진입 조건 충족
            if pred_probs[i] >= final_th:
                # 수익률 계산 (순수 수익률 - 왕복 비용)
                raw_return = returns_arr[i]
                net_return = raw_return - round_trip_cost

                executed_trades.append(net_return)

                # 진입했으므로 쿨다운 적용
                next_trade_idx = i + cooldown

        # ---------------------------------------------------------
        # Step 5: 최종 성과 분석
        # ---------------------------------------------------------
        if not executed_trades:
            print("\n>>> [Warning] 조건에 맞는 진입 신호가 없습니다. (No Trades)")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # 승률
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades if n_trades > 0 else 0

        # 수익률
        avg_return = np.mean(executed_trades)
        cum_return = np.prod(executed_trades + 1) - 1
        std_return = np.std(executed_trades)

        # Sharpe Ratio (단순화: 무위험이자율 0 가정)
        sharpe_ratio = (avg_return / std_return) * np.sqrt(n_trades) if std_return > 0 else 0

        # MDD (Maximum Drawdown)
        cumulative_returns = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cumulative_returns)
        drawdown = (cumulative_returns - peak) / peak
        max_drawdown = drawdown.min()

        # 손익비 (Profit Factor 유사 개념)
        avg_win = win_trades.mean() if len(win_trades) > 0 else 0
        avg_loss = loss_trades.mean() if len(loss_trades) > 0 else 0
        pnl_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')

        print("-" * 40)
        print(f" [최종 백테스트 결과 (N={n_trades})]")
        print("-" * 40)
        print(f" 1. 승률 (Win Rate)       : {win_rate * 100:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_return * 100:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_return * 100:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe_ratio:.4f}")
        print(f" 5. MDD (Max Drawdown)   : {max_drawdown * 100:.2f}%")
        print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        print("-" * 40)


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()
    bot.apply_labeling()
    bot.split_and_scale()
    bot.run_optuna()
    model = bot.train_final_models()
    bot.evaluate()

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (516465, 92)
[2] Triple Barrier 레이블링 (Cost-aware) 적용 중...
    >>> round_trip_cost: 0.090% (fee=0.025%, slip=0.020%)
    >>> MIN_RET (success 기준): 0.185%
    >>> 레이블링 완료. 상승(1) 비율: 0.51%
[3] 데이터 분할 및 스케일링...


[I 2025-12-02 07:42:44,397] A new study created in memory with name: no-name-da3acf65-ad17-4b00-ad9e-1fa0f26957b6


    >>> 학습 셋: (412980, 86)
[4] 하이퍼파라미터 튜닝 (Target: Accuracy, Trials: 5)


[I 2025-12-02 07:43:03,771] Trial 0 finished with value: 0.9958593636495714 and parameters: {'fl_alpha': 0.796536029782884, 'fl_gamma': 4.489665534951962, 'n_estimators': 437, 'max_depth': 6, 'learning_rate': 0.05051665723685949, 'subsample': 0.7137113279395896, 'colsample_bytree': 0.6583344746361799, 'reg_alpha': 0.005350845491616897, 'reg_lambda': 0.003119282057005311, 'xgb_gamma': 0.2542989861372056}. Best is trial 0 with value: 0.9958593636495714.
[I 2025-12-02 07:43:29,189] Trial 1 finished with value: 0.9958593636495714 and parameters: {'fl_alpha': 0.6588199917724078, 'fl_gamma': 2.796228588471817, 'n_estimators': 635, 'max_depth': 5, 'learning_rate': 0.05676987551961453, 'subsample': 0.856178234942099, 'colsample_bytree': 0.6788132477488362, 'reg_alpha': 0.002188708888647921, 'reg_lambda': 0.009762854745716155, 'xgb_gamma': 0.7092409831701352}. Best is trial 0 with value: 0.9958593636495714.
[I 2025-12-02 07:43:50,743] Trial 2 finished with value: 0.995441667877379 and parameter

    >>> Best Params: {'fl_alpha': 0.796536029782884, 'fl_gamma': 4.489665534951962, 'n_estimators': 437, 'max_depth': 6, 'learning_rate': 0.05051665723685949, 'subsample': 0.7137113279395896, 'colsample_bytree': 0.6583344746361799, 'reg_alpha': 0.005350845491616897, 'reg_lambda': 0.003119282057005311, 'xgb_gamma': 0.2542989861372056}
[5] 최종 모델 학습...
    >>> 모델 학습 완료 (Alpha=0.80, Gamma=4.49)

 [ 전략 평가 및 백테스팅 (Flexible Threshold) ]
>>> 적용 왕복 비용: 0.090%

[Diagnosis] 예측 확률 분포 요약:
count    103246.0
mean          0.5
std           0.0
min           0.5
50%           0.5
75%           0.5
90%           0.5
95%           0.5
99%           0.5
max           0.5
dtype: float64

[Optimization] F1-Score 최대화 임계값 발견
    >>> Max F1: 0.0024 at Threshold: 0.5000
    >>> 최종 적용 Threshold: 0.5000

[Backtest] 시뮬레이션 시작 (쿨다운: 40 캔들)...
----------------------------------------
 [최종 백테스트 결과 (N=2582)]
----------------------------------------
 1. 승률 (Win Rate)       : 43.69%
 2. 평균 수익률 (Avg Ret) : -0.0888%
 3. 누

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score) # f1_score 추가
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 사용자 조정 파라미터 (Moderate Strategy)
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # 거래 비용
    'fee_rate': 0.00025,
    'slippage': 0.00020,

    'triple_barrier': {
        'span': 100,            # 변동성 계산 기간
        # [사용자 지정] 익절 배수: 2.2 -> 2.3 (소폭 상향)
        'pt': 1.4,
        # [기본 유지] 손절 배수: 1.0 (방어적 설정 유지)
        'sl': 0.7,
        # [사용자 지정] 최대 보유 기간: 10 -> 30 (중기 추세 반영)
        'vertical_barrier': 15
    },
    'optuna_trials': 10,
    'purge_gap': 10,
    # [사용자 지정] 진입 장벽: 0.60 (방어적 고정)
    'meta_labeling_threshold': 0.55
}

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None

        # 데이터셋
        self.X_train = None
        self.X_test = None
        self.y_class_train = None  # 분류 타겟 (0, 1)
        self.y_class_test = None

        # 검증용 수익률 데이터 (학습엔 안 씀)
        self.y_return_test = None

        # 모델
        self.classifier = None

        self.feature_cols = []
        self.best_params_class = {}

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        try:
            self.df = pd.read_csv(self.config['file_path'])
            self.df['datetime'] = pd.to_datetime(self.df['datetime'])
            self.df = self.df.set_index('datetime').sort_index()
            self.df.dropna(inplace=True)
            print(f"    >>> 데이터 로드 완료: {self.df.shape}")
        except Exception as e:
            print(f"    [Error] 데이터 로드 실패: {e}")

    # =============================================================================
    # 2. Triple Barrier (Cost-aware Labeling 적용)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        print("[2] Triple Barrier 레이블링 (Cost-aware) 적용 중...")

        # 1) 변동성 계산
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        # 2) Triple Barrier 파라미터
        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        # 3) 거래 비용 기반 최소 요구 수익률 계산
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00025)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # 최소 마진 설정
        MIN_RET = round_trip_cost * 1.5 + 0.0005

        print(f"    >>> round_trip_cost: {round_trip_cost*100:.3f}%")
        print(f"    >>> MIN_RET (success 기준): {MIN_RET*100:.3f}%")

        labels = []
        returns = []

        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values

        n_samples = len(closes)

        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.001)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = np.nan
            ret = np.nan
            touched = False

            for j in range(1, t_final + 1):
                # Take Profit
                if highs[i + j] >= upper:
                    ret = (upper - current_price) / current_price
                    if ret > MIN_RET:
                        label = 1
                    else:
                        label = 0
                    touched = True
                    break

                # Stop Loss
                if lows[i + j] <= lower:
                    ret = (lower - current_price) / current_price
                    label = 0
                    touched = True
                    break

            # Time Limit
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price
                if ret > MIN_RET:
                    label = 1
                else:
                    label = 0

            labels.append(label)
            returns.append(ret)

        # Padding
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns
        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화
    # =============================================================================
    def split_and_scale(self):
        print("[3] 데이터 분할 및 스케일링 (Rolling Scaling)...")

        exclude_cols = ['target_class', 'target_return', 'open', 'high', 'low', 'close', 'volume', 'value', 'volatility', 'datetime']
        self.feature_cols = [col for col in self.df.columns if col not in exclude_cols]

        if not self.feature_cols:
            self.feature_cols = ['open', 'high', 'low', 'volume']

        X = self.df[self.feature_cols]

        # Rolling Scaling
        window = 1000
        rolling_mean = X.rolling(window=window).mean()
        rolling_std = X.rolling(window=window).std()
        rolling_std = rolling_std.replace(0, 1)
        X_scaled = (X - rolling_mean) / rolling_std

        valid_idx = X_scaled.dropna().index
        X_scaled = X_scaled.loc[valid_idx]
        y_c = self.df.loc[valid_idx, 'target_class']
        y_ret = self.df.loc[valid_idx, 'target_return']

        split_point = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_point]
        self.X_test = X_scaled.iloc[split_point:]
        self.y_class_train = y_c.iloc[:split_point]
        self.y_class_test = y_c.iloc[split_point:]
        self.y_return_test = y_ret.iloc[split_point:]

        # Clipping
        self.X_train = self.X_train.clip(-5, 5)
        self.X_test = self.X_test.clip(-5, 5)
        print(f"    >>> 학습 셋: {self.X_train.shape}")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    # _purged_cv_score 함수 전체를 이렇게 바꾸세요
    def _purged_cv_score(self, model, X, y, n_splits=3):
        kfold = KFold(n_splits=n_splits, shuffle=False)
        scores = []
        gap = self.config['purge_gap']

        for train_idx, val_idx in kfold.split(X):
            val_start = val_idx[0]
            val_end = val_idx[-1]
            real_train_idx = [i for i in train_idx if i < val_start - gap or i > val_end + gap]
            if len(real_train_idx) < 100:
                continue

            X_tr = X.iloc[real_train_idx]
            y_tr = y.iloc[real_train_idx]
            X_val = X.iloc[val_idx]
            y_val = y.iloc[val_idx]

            model.fit(X_tr, y_tr, verbose=False)
            pred = model.predict(X_val)                    # ← 그냥 0.5 기준 예측
            scores.append(accuracy_score(y_val, pred))     # ← Accuracy로 변경

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        neg = (self.y_class_train == 0).sum()
        pos = (self.y_class_train == 1).sum()
        imbalance_ratio = neg / pos if pos > 0 else 1.0
        print(f"    >>> 데이터 불균형 비율: {imbalance_ratio:.2f}")

        def objective_cls(trial):
            # [수정] 튜닝 범위를 넓혀서 Optuna가 최적의 파라미터를 찾을 기회를 늘립니다.
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 800), # 600 -> 800
                'max_depth': trial.suggest_int('max_depth', 3, 15),          # 12 -> 15
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15), # 0.01 -> 0.005, 0.1 -> 0.15
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),     # 0.95 -> 1.0
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0), # 0.95 -> 1.0
                'gamma': trial.suggest_float('gamma', 0, 8),                 # 5 -> 8
                # scale_pos_weight 범위도 약간 넓힘
                'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, imbalance_ratio + 5.0),
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        # [수정] Optuna의 방향을 F1 Score에 맞게 'maximize'로 설정합니다.
        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])
        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")

    # =============================================================================
    # 6. 최종 학습 및 평가 (Non-overlapping Backtest)
    # =============================================================================
    def _evaluate(self):
        # 1. 확률 예측
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]

        print("\n" + "="*60)
        print(" [ Meta Labeling 전략 평가 (Non-overlapping / Net Return) ]")
        print("="*60)

        threshold = self.config['meta_labeling_threshold']
        vertical_barrier = self.config['triple_barrier']['vertical_barrier']

        # [비용 계산]
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00020)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        print(f">>> 적용 Threshold: {threshold}")
        print(f">>> 적용 왕복 비용: {round_trip_cost*100:.3f}%")

        # 2. 중복 방지 백테스팅
        returns_arr = self.y_return_test.values
        executed_trades = []
        next_trade_idx = 0

        for i in range(len(pred_probs)):
            if i < next_trade_idx:
                continue

            if pred_probs[i] > threshold:
                # [수정] 비용 차감 로직 적용
                gross_ret = returns_arr[i]
                net_ret = gross_ret - round_trip_cost  # 비용 차감

                executed_trades.append(net_ret)

                # 진입 후 보유기간 동안 신규 진입 금지 (잘 구현됨)
                next_trade_idx = i + vertical_barrier

        # 3. 결과 집계
        if not executed_trades:
            print(">>> [Warning] 진입 신호 없음. Threshold를 더 낮추세요.")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # 승률 (0보다 큰 수익이 난 경우)
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades if n_trades > 0 else 0

        avg_return = np.mean(executed_trades)
        cum_return = np.prod(executed_trades + 1) - 1

        # 샤프 지수 (간이)
        std_return = np.std(executed_trades)
        sharpe = (avg_return / std_return) * np.sqrt(n_trades) if std_return > 0 else 0

        print(f"    - 총 진입 횟수: {n_trades}회")
        print(f"    - 승률 (Win Rate): {win_rate*100:.2f}%")
        print(f"    - 평균 수익률 (Avg Net Ret): {avg_return*100:.4f}% (비용 차감 후)")
        print(f"    - 누적 수익률 (Cumulative): {cum_return*100:.2f}%")
        print(f"    - 샤프 지수: {sharpe:.4f}")

        if len(loss_trades) > 0:
            avg_win = win_trades.mean() if len(win_trades) > 0 else 0
            avg_loss = loss_trades.mean()
            pnl_ratio = abs(avg_win / avg_loss)
            print(f"    - 손익비 (P/L Ratio): {pnl_ratio:.2f}")

if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()
    bot.apply_labeling()
    bot.split_and_scale()
    bot.run_optuna()
    cls_model = bot.train_final_models()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(cls_model, f"btc_model_moderate_{timestamp}.pkl")
    print(f"\n>>> 모델 저장 완료 (btc_model_moderate_{timestamp}.pkl)")

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (516465, 92)
[2] Triple Barrier 레이블링 (Cost-aware) 적용 중...
    >>> round_trip_cost: 0.090%
    >>> MIN_RET (success 기준): 0.185%
    >>> 레이블링 완료. 상승(1) 비율: 1.15%
[3] 데이터 분할 및 스케일링 (Rolling Scaling)...


[I 2025-11-29 08:39:08,921] A new study created in memory with name: no-name-72281482-3a37-4997-9189-b5ed64541ff7


    >>> 학습 셋: (412360, 85)
[4] 하이퍼파라미터 튜닝 (Trials: 10)
    >>> 데이터 불균형 비율: 71.24


[W 2025-11-29 08:42:20,864] Trial 0 failed with parameters: {'n_estimators': 681, 'max_depth': 14, 'learning_rate': 0.014826254714057442, 'subsample': 0.7362236374183643, 'colsample_bytree': 0.9068962812756609, 'gamma': 1.9944526175238195, 'scale_pos_weight': 31.198872161434977} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-1414261835.py", line 262, in objective_cls
    return self._purged_cv_score(model, self.X_train, self.y_class_train)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1414261835.py", line 230, in _purged_cv_score
    pred = model.predict(X_val)                    # ← 그냥 0.5 기준 예측
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xgboost/core.py", line 77

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score) # f1_score 추가
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 사용자 조정 파라미터 (Moderate Strategy)
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # 거래 비용
    'fee_rate': 0.00025,
    'slippage': 0.00020,

    'triple_barrier': {
        'span': 100,            # 변동성 계산 기간
        # [사용자 지정] 익절 배수: 2.2 -> 2.3 (소폭 상향)
        'pt': 0.7,
        # [기본 유지] 손절 배수: 1.0 (방어적 설정 유지)
        'sl': 1.5,
        # [사용자 지정] 최대 보유 기간: 10 -> 30 (중기 추세 반영)
        'vertical_barrier': 15
    },
    'optuna_trials': 5,
    'purge_gap': 10,
    # [사용자 지정] 진입 장벽: 0.60 (방어적 고정)
    'meta_labeling_threshold': 0.50
}

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None

        # 데이터셋
        self.X_train = None
        self.X_test = None
        self.y_class_train = None  # 분류 타겟 (0, 1)
        self.y_class_test = None

        # 검증용 수익률 데이터 (학습엔 안 씀)
        self.y_return_test = None

        # 모델
        self.classifier = None

        self.feature_cols = []
        self.best_params_class = {}

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        try:
            self.df = pd.read_csv(self.config['file_path'])
            self.df['datetime'] = pd.to_datetime(self.df['datetime'])
            self.df = self.df.set_index('datetime').sort_index()
            self.df.dropna(inplace=True)
            print(f"    >>> 데이터 로드 완료: {self.df.shape}")
        except Exception as e:
            print(f"    [Error] 데이터 로드 실패: {e}")

    # =============================================================================
    # 2. Triple Barrier (Cost-aware Labeling 적용)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        print("[2] Triple Barrier 레이블링 (Cost-aware) 적용 중...")

        # 1) 변동성 계산
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        # 2) Triple Barrier 파라미터
        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        # 3) 거래 비용 기반 최소 요구 수익률 계산
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00025)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # 최소 마진 설정
        MIN_RET = round_trip_cost * 1.5 + 0.0005

        print(f"    >>> round_trip_cost: {round_trip_cost*100:.3f}%")
        print(f"    >>> MIN_RET (success 기준): {MIN_RET*100:.3f}%")

        labels = []
        returns = []

        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values

        n_samples = len(closes)

        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.001)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = np.nan
            ret = np.nan
            touched = False

            for j in range(1, t_final + 1):
                # Take Profit
                if highs[i + j] >= upper:
                    ret = (upper - current_price) / current_price
                    if ret > MIN_RET:
                        label = 1
                    else:
                        label = 0
                    touched = True
                    break

                # Stop Loss
                if lows[i + j] <= lower:
                    ret = (lower - current_price) / current_price
                    label = 0
                    touched = True
                    break

            # Time Limit (시간 만료 시)
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price

                # [수정 전] MIN_RET (비용 포함 마진) 이상이어야 1
                # if ret > MIN_RET:

                # [수정 후] 비용만 건져도 '성공(1)'으로 간주
                # 이유: 손실이 아닌 거래를 모델에게 '나쁜 거래'라고 가르치면
                # 모델이 너무 소극적으로 변함. 본전 이상이면 일단 1로 줘서 진입을 유도.
                if ret > 0:  # 0보다만 크면 성공으로 라벨링 (Fee 고려는 백테스트에서 함)
                    label = 1
                else:
                    label = 0

            labels.append(label)
            returns.append(ret)

        # Padding
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns
        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화
    # =============================================================================
    def split_and_scale(self):
        print("[3] 데이터 분할 및 스케일링 (Rolling Scaling)...")

        exclude_cols = ['target_class', 'target_return', 'open', 'high', 'low', 'close', 'volume', 'value', 'volatility', 'datetime']
        self.feature_cols = [col for col in self.df.columns if col not in exclude_cols]

        if not self.feature_cols:
            self.feature_cols = ['open', 'high', 'low', 'volume']

        X = self.df[self.feature_cols]

        # Rolling Scaling
        window = 1000
        rolling_mean = X.rolling(window=window).mean()
        rolling_std = X.rolling(window=window).std()
        rolling_std = rolling_std.replace(0, 1)
        X_scaled = (X - rolling_mean) / rolling_std

        valid_idx = X_scaled.dropna().index
        X_scaled = X_scaled.loc[valid_idx]
        y_c = self.df.loc[valid_idx, 'target_class']
        y_ret = self.df.loc[valid_idx, 'target_return']

        split_point = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_point]
        self.X_test = X_scaled.iloc[split_point:]
        self.y_class_train = y_c.iloc[:split_point]
        self.y_class_test = y_c.iloc[split_point:]
        self.y_return_test = y_ret.iloc[split_point:]

        # Clipping
        self.X_train = self.X_train.clip(-5, 5)
        self.X_test = self.X_test.clip(-5, 5)
        print(f"    >>> 학습 셋: {self.X_train.shape}")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    # 맨 위 임포트 부분

    # _purged_cv_score 함수 전체 교체
    def _purged_cv_score(self, model, X, y, n_splits=3):
        kfold = KFold(n_splits=n_splits, shuffle=False)
        scores = []
        gap = self.config['purge_gap']

        for train_idx, val_idx in kfold.split(X):
            val_start = val_idx[0]
            val_end = val_idx[-1]
            real_train_idx = [i for i in train_idx if i < val_start - gap or i > val_end + gap]
            if len(real_train_idx) < 100:
                continue

            X_tr = X.iloc[real_train_idx]
            y_tr = y.iloc[real_train_idx]
            X_val = X.iloc[val_idx]
            y_val = y.iloc[val_idx]

            model.fit(X_tr, y_tr, verbose=False)
            pred = model.predict(X_val)
            scores.append(accuracy_score(y_val, pred))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        neg = (self.y_class_train == 0).sum()
        pos = (self.y_class_train == 1).sum()
        imbalance_ratio = neg / pos if pos > 0 else 1.0
        print(f"    >>> 데이터 불균형 비율: {imbalance_ratio:.2f}")

        def objective_cls(trial):
            # [수정] 모델 과적합 및 가중치 불균형을 방지하기 위한 파라미터 제약
            params = {
                # 1. 깊이 제한: 과적합 방지를 위해 3~7 사이로 제한
                'max_depth': trial.suggest_int('max_depth', 3, 7),

                # 2. 가중치 제한: 너무 공격적인 예측 방지를 위해 1.0 ~ 3.0으로 억제
                'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 3.0),

                # 3. 학습률: 안정적인 학습을 위해 범위 조정
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),

                # 나머지 파라미터
                'n_estimators': trial.suggest_int('n_estimators', 100, 600),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'gamma': trial.suggest_float('gamma', 1, 5),
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        # Optuna의 방향을 'maximize'로 설정 (Purged CV Score/Accuracy 극대화)
        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])
        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")

    # =============================================================================
    # 6. 최종 학습 및 평가 (Non-overlapping Backtest)
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습 및 검증...")

        if not self.best_params_class:
             self.best_params_class = {
                'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05,
                'n_jobs': -1, 'random_state': 42
            }

        self.classifier = xgb.XGBClassifier(**self.best_params_class)
        self.classifier.fit(self.X_train, self.y_class_train)

        self._evaluate()
        return self.classifier

    def _evaluate(self):
        # 1. 확률 예측
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]

        print("\n" + "="*60)
        print(" [ Meta Labeling 전략 평가 (Realistic & Dynamic Skip) ]")
        print("="*60)

        threshold = self.config['meta_labeling_threshold']

        # Triple Barrier 파라미터 가져오기
        vertical_barrier = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        # 비용 계산
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00020)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        print(f">>> 적용 Threshold: {threshold}")
        print(f">>> 적용 왕복 비용: {round_trip_cost*100:.3f}%")
        print(f">>> 검증 로직: Low 우선 체크 (보수적) + 조기 청산 시 즉시 재진입")

        # 테스트 데이터의 원본 가격 정보 가져오기 (High/Low/Close)
        # X_test 인덱스에 해당하는 원본 df 데이터 매핑
        test_indices = self.X_test.index
        # 원본 데이터에서 해당 구간 슬라이싱
        price_data = self.df.loc[test_indices]

        closes = price_data['close'].values
        highs = price_data['high'].values
        lows = price_data['low'].values
        vols = price_data['volatility'].values # 변동성 기반 배리어일 경우

        executed_trades = []
        next_trade_idx = 0

        # -----------------------------------------------------------------
        # [수정] 백테스트 시뮬레이션 루프
        # -----------------------------------------------------------------
        for i in range(len(pred_probs)):
            # 1. 이미 진입한 포지션이 있으면 패스 (Cooldown)
            if i < next_trade_idx:
                continue

            # 2. 진입 신호 발생
            if pred_probs[i] >= threshold:
                entry_price = closes[i]
                current_vol = vols[i]

                # 동적 배리어 설정 (ATR/Std 기반)
                # 만약 고정 %라면 current_vol 대신 고정값 사용
                # 여기서는 코드의 로직에 따라 vol 기반이라 가정
                limit_vol = max(current_vol, 0.001)

                target_price = entry_price * (1 + limit_vol * pt)
                stop_price = entry_price * (1 - limit_vol * sl)

                trade_ret = 0.0
                holding_period = vertical_barrier # 기본은 만기까지 보유

                # 3. 진입 후 캔들 하나씩 확인 (Triple Barrier Simulation)
                # i+1 부터 i+vertical_barrier 까지
                for j in range(1, vertical_barrier + 1):
                    if i + j >= len(closes):
                        holding_period = j
                        break # 데이터 끝

                    curr_high = highs[i + j]
                    curr_low = lows[i + j]

                    # [중요 1] 보수적 검증: Low(손절) 먼저 체크!
                    if curr_low <= stop_price:
                        # 손절 발생
                        gross_ret = (stop_price - entry_price) / entry_price
                        trade_ret = gross_ret - round_trip_cost
                        holding_period = j # 실제 보유 기간 업데이트
                        break # 포지션 종료

                    # [중요 1] 그 다음 High(익절) 체크
                    elif curr_high >= target_price:
                        # 익절 발생
                        gross_ret = (target_price - entry_price) / entry_price
                        trade_ret = gross_ret - round_trip_cost
                        holding_period = j # 실제 보유 기간 업데이트
                        break # 포지션 종료

                    # 만약 j가 마지막(vertical_barrier)이면 타임아웃 청산
                    if j == vertical_barrier:
                        exit_price = closes[i + j]
                        gross_ret = (exit_price - entry_price) / entry_price
                        trade_ret = gross_ret - round_trip_cost
                        holding_period = j

                # 거래 기록 저장
                executed_trades.append(trade_ret)

                # [중요 2] 조기 청산 반영: 실제 보유 기간만큼만 건너뜀
                next_trade_idx = i + holding_period

        # -----------------------------------------------------------------
        # 결과 집계
        # -----------------------------------------------------------------
        if not executed_trades:
            print(">>> [Warning] 진입 신호 없음.")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades if n_trades > 0 else 0

        avg_return = np.mean(executed_trades)
        cum_return = np.prod(executed_trades + 1) - 1

        std_return = np.std(executed_trades)
        sharpe = (avg_return / std_return) * np.sqrt(n_trades) if std_return > 0 else 0

        # MDD
        cum_series = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cum_series)
        dd = (cum_series - peak) / peak
        max_mdd = dd.min()

        print("-" * 40)
        print(f" [최종 백테스트 결과 (N={n_trades})]")
        print("-" * 40)
        print(f" 1. 승률 (Win Rate)       : {win_rate * 100:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_return * 100:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_return * 100:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe:.4f}")
        print(f" 5. MDD (Max Drawdown)    : {max_mdd * 100:.2f}%")

        if len(loss_trades) > 0:
            avg_win = win_trades.mean() if len(win_trades) > 0 else 0
            avg_loss = loss_trades.mean()
            pnl_ratio = abs(avg_win / avg_loss)
            print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        else:
            print(f" 6. 손익비 (P/L Ratio)    : Inf")
        print("-" * 40)

if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()
    bot.apply_labeling()
    bot.split_and_scale()
    bot.run_optuna()
    cls_model = bot.train_final_models()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(cls_model, f"btc_model_moderate_{timestamp}.pkl")
    print(f"\n>>> 모델 저장 완료 (btc_model_moderate_{timestamp}.pkl)")

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (516465, 92)
[2] Triple Barrier 레이블링 (Cost-aware) 적용 중...
    >>> round_trip_cost: 0.090%
    >>> MIN_RET (success 기준): 0.185%
    >>> 레이블링 완료. 상승(1) 비율: 8.99%
[3] 데이터 분할 및 스케일링 (Rolling Scaling)...


[I 2025-11-29 12:05:04,720] A new study created in memory with name: no-name-cf598188-6697-4fe5-824d-9b7dac3713f2


    >>> 학습 셋: (412360, 85)
[4] 하이퍼파라미터 튜닝 (Trials: 5)
    >>> 데이터 불균형 비율: 11.58


[I 2025-11-29 12:05:27,485] Trial 0 finished with value: 0.9203777544468954 and parameters: {'max_depth': 5, 'scale_pos_weight': 1.1855087499935018, 'learning_rate': 0.02590891581252902, 'n_estimators': 584, 'subsample': 0.7778726165736993, 'colsample_bytree': 0.8668922430697098, 'gamma': 3.1108611991628745}. Best is trial 0 with value: 0.9203777544468954.
[I 2025-11-29 12:05:48,067] Trial 1 finished with value: 0.9197472367936691 and parameters: {'max_depth': 5, 'scale_pos_weight': 1.6320524979372493, 'learning_rate': 0.09310348834021713, 'n_estimators': 447, 'subsample': 0.88063390087519, 'colsample_bytree': 0.6481818628034562, 'gamma': 2.713187627776579}. Best is trial 0 with value: 0.9203777544468954.
[I 2025-11-29 12:06:05,765] Trial 2 finished with value: 0.9200091444473749 and parameters: {'max_depth': 5, 'scale_pos_weight': 2.825864083056046, 'learning_rate': 0.010342983925042563, 'n_estimators': 287, 'subsample': 0.7272716026354802, 'colsample_bytree': 0.7410025551152504, 'gam

    >>> Best Params: {'max_depth': 5, 'scale_pos_weight': 1.1855087499935018, 'learning_rate': 0.02590891581252902, 'n_estimators': 584, 'subsample': 0.7778726165736993, 'colsample_bytree': 0.8668922430697098, 'gamma': 3.1108611991628745}
[5] 최종 모델 학습 및 검증...

 [ Meta Labeling 전략 평가 (Realistic & Dynamic Skip) ]
>>> 적용 Threshold: 0.5
>>> 적용 왕복 비용: 0.090%
>>> 검증 로직: Low 우선 체크 (보수적) + 조기 청산 시 즉시 재진입
>>> [Warning] 진입 신호 없음.

>>> 모델 저장 완료 (btc_model_moderate_20251129_120743.pkl)


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (classification_report, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
import optuna
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 하이퍼파라미터 및 경로 설정
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,
    'triple_barrier': {
        'span': 100,           # 변동성 계산 기간
        'pt': 1.5,             # 익절 배수
        'sl': 1.5,             # 손절 배수
        'vertical_barrier': 15 # 최대 보유 기간 (캔들 수)
    },
    'optuna_trials': 10,
    'purge_gap': 10,
    'meta_labeling_threshold': 0.60
}

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None

        # 데이터셋
        self.X_train = None
        self.X_test = None
        self.y_class_train = None  # 분류 타겟 (0, 1)
        self.y_class_test = None

        # 검증용 수익률 데이터 (학습엔 안 씀)
        self.y_return_test = None

        # 모델
        self.classifier = None

        self.feature_cols = []
        self.best_params_class = {}

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        try:
            self.df = pd.read_csv(self.config['file_path'])
            self.df['datetime'] = pd.to_datetime(self.df['datetime'])
            self.df = self.df.set_index('datetime').sort_index()
            self.df.dropna(inplace=True)
            print(f"    >>> 데이터 로드 완료: {self.df.shape}")
        except Exception as e:
            print(f"    [Error] 데이터 로드 실패: {e}")
            print("    [Info] 더미 데이터를 생성하여 진행합니다.")
            dates = pd.date_range(start='2023-01-01', periods=5000, freq='1T')
            self.df = pd.DataFrame({
                'open': np.random.rand(5000) * 100 + 20000,
                'high': np.random.rand(5000) * 100 + 20100,
                'low': np.random.rand(5000) * 100 + 19900,
                'close': np.random.rand(5000) * 100 + 20000,
                'volume': np.random.rand(5000) * 1000
            }, index=dates)
            self.df['volatility'] = self.df['close'].pct_change().rolling(20).std()
            self.df.dropna(inplace=True)

    # =============================================================================
    # 2. Triple Barrier (분류 라벨 생성)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        print("[2] Triple Barrier 레이블링 (Logic Improved) 적용 중...")

        self.df['volatility'] = self._get_volatility(self.df['close'], span=self.config['triple_barrier']['span'])

        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        labels = []  # 0: 실패/손절/횡보, 1: 성공/익절
        returns = [] # 실제 수익률 (검증용)

        closes = self.df['close'].values
        vols = self.df['volatility'].values
        highs = self.df['high'].values
        lows = self.df['low'].values

        n_samples = len(closes)

        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.001)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = np.nan
            ret = np.nan
            touched = False

            # 1. Triple Barrier 수정 예시 (close-only 버전, 가장 보수적)
            for j in range(1, t_final + 1):
                future_close = closes[i + j]
                if future_close >= upper:
                    label = 1
                    ret = (upper - current_price) / current_price  # 실제 체결 가격으로 보정
                    break
                if future_close <= lower:
                    label = 0
                    ret = (lower - current_price) / current_price
                    break
            else:
                # vertical barrier
                ret = (closes[i + t_final] - current_price) / current_price
                label = 1 if ret > 0.002 else 0  # 더 엄격하게
                # vertical barrier
                ret = (closes[i + t_final] - current_price) / current_price
                label = 1 if ret > 0.002 else 0  # 더 엄격하게

            if not touched:
                # Vertical Barrier (시간 초과)
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price

                # [수정] 최소 수익률 0.1% (수수료 고려) 이상이어야만 성공(1)으로 간주
                # 기존에는 > 0 이면 무조건 1이었으나, 이는 노이즈를 너무 많이 포함함.
                if ret > 0.001:
                    label = 1
                else:
                    label = 0

            labels.append(label)
            returns.append(ret)

        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns # 검증용

        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화 (Rolling Scaling)
    # =============================================================================
    def split_and_scale(self):
        print("[3] 데이터 분할 및 스케일링 (Rolling Scaling)...")

        exclude_cols = ['target_class', 'target_return', 'open', 'high', 'low', 'close', 'volume', 'value', 'volatility', 'datetime']
        self.feature_cols = [col for col in self.df.columns if col not in exclude_cols]

        if not self.feature_cols:
            self.feature_cols = ['open', 'high', 'low', 'volume']

        X = self.df[self.feature_cols]

        window = 1000
        rolling_mean = X.rolling(window=window).mean()
        rolling_std = X.rolling(window=window).std()
        rolling_std = rolling_std.replace(0, 1)

        X_scaled = (X - rolling_mean) / rolling_std

        valid_idx = X_scaled.dropna().index
        X_scaled = X_scaled.loc[valid_idx]
        y_c = self.df.loc[valid_idx, 'target_class']
        y_ret = self.df.loc[valid_idx, 'target_return']

        split_point = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_point]
        self.X_test = X_scaled.iloc[split_point:]

        self.y_class_train = y_c.iloc[:split_point]
        self.y_class_test = y_c.iloc[split_point:]

        # 검증용 수익률 (Test set만 필요)
        self.y_return_test = y_ret.iloc[split_point:]

        # 클리핑
        self.X_train = self.X_train.clip(-5, 5)
        self.X_test = self.X_test.clip(-5, 5)

        print(f"    >>> 학습 셋: {self.X_train.shape}")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    def _purged_cv_score(self, model, X, y, n_splits=3):
        kfold = KFold(n_splits=n_splits, shuffle=False)
        scores = []
        gap = self.config['purge_gap']

        for train_idx, val_idx in kfold.split(X):
            val_start = val_idx[0]
            val_end = val_idx[-1]

            real_train_idx = [i for i in train_idx if i < val_start - gap or i > val_end + gap]

            if len(real_train_idx) < 100: continue

            X_tr = X.iloc[real_train_idx]
            y_tr = y.iloc[real_train_idx]
            X_val = X.iloc[val_idx]
            y_val = y.iloc[val_idx]

            model.fit(X_tr, y_tr, verbose=False)
            pred = model.predict(X_val)

            scores.append(f1_score(y_val, pred, average='macro'))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝 (Classification Only)
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        # 불균형 비율 계산 (참고용 Max 값)
        neg = (self.y_class_train == 0).sum()
        pos = (self.y_class_train == 1).sum()
        imbalance_ratio = neg / pos if pos > 0 else 1.0

        print(f"    >>> 데이터 불균형 비율(Neg/Pos): {imbalance_ratio:.2f}")
        print("    >>> [수정] scale_pos_weight 고정값 제거 -> Optuna 튜닝 파라미터로 전환")

        def objective_cls(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'gamma': trial.suggest_float('gamma', 0, 5),

                # [수정] scale_pos_weight를 고정하지 않고 튜닝 (1.0 ~ 비율의 1.2배 사이)
                # 1.0에 가까우면 Precision 중시, 비율에 가까우면 Recall 중시
                'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, imbalance_ratio + 2.0),

                'n_jobs': -1,
                'random_state': self.config['random_state']
            }
            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])
        self.best_params_class = study_cls.best_params

        print(f"    >>> Best Params: {self.best_params_class}")
        print("    >>> 튜닝 완료.")

    # =============================================================================
    # 6. 최종 모델 학습 및 Meta Labeling 검증
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습 및 Meta Labeling 검증...")

        self.classifier = xgb.XGBClassifier(**self.best_params_class, n_jobs=-1)
        self.classifier.fit(self.X_train, self.y_class_train)

        self._evaluate()

        return self.classifier

    def _evaluate(self):
        """
        수정된 평가 로직:
        1. 예측 확률 분포를 먼저 확인하여 Threshold가 현실적인지 진단합니다.
        2. 지나치게 높은 Threshold는 상위 N% 수준으로 자동 보정 제안을 합니다.
        3. 진입 후 대기 시간(Cooldown)을 줄여 거래 기회를 확보합니다.
        """
        # 1. 모델 예측 (확률값 추출)
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]

        print("\n" + "="*60)
        print(" [ Meta Labeling 전략 평가 (Improved) ]")
        print("="*60)

        # ---------------------------------------------------------
        # [개선 1] 예측 확률 분포 진단 (모델이 얼마나 확신하는가?)
        # ---------------------------------------------------------
        probs_series = pd.Series(pred_probs)
        print("\n[Diagnosis] 모델 예측 확률 분포 요약:")
        print(probs_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

        # ---------------------------------------------------------
        # [개선 2] Threshold 동적 설정 및 유효성 검사
        # ---------------------------------------------------------
        target_threshold = self.config['meta_labeling_threshold']

        # 만약 설정한 Threshold(0.7)가 모델 예측의 최대값보다 크다면? (거래 0회 방지)
        max_prob = probs_series.max()
        if target_threshold > max_prob:
            print(f"\n>>> [Alert] 설정한 Threshold({target_threshold})가 모델 예측 최대값({max_prob:.4f})보다 높습니다.")
            # 상위 5% 수준으로 강제 조정 (최소 0.5는 넘겨야 함)
            new_threshold = max(0.51, probs_series.quantile(0.95))
            print(f">>> Threshold를 상위 5% 수준인 {new_threshold:.4f}로 자동 조정합니다.")
            target_threshold = new_threshold

        # 분포상 상위 10% 정도는 되어야 의미가 있음
        top_10_percent = probs_series.quantile(0.90)
        if target_threshold < top_10_percent:
             print(f">>> (Info) 현재 Threshold({target_threshold})는 상위 10%({top_10_percent:.4f})보다 낮아 진입이 많을 수 있습니다.")

        print(f"\n>>> 최종 적용 Threshold: {target_threshold:.4f}")

        # ---------------------------------------------------------
        # [개선 3] 백테스팅 로직 (Cooldown 완화)
        # ---------------------------------------------------------
        # 기존에는 vertical_barrier(30) 만큼 무조건 쉬었으나,
        # 이를 줄여서(예: 5분) 연속적인 기회를 포착하도록 변경
        cooldown = 5
        print(f">>> 진입 후 쿨다운(대기): {cooldown} 캔들 (기존: {self.config['triple_barrier']['vertical_barrier']})")

        returns_arr = self.y_return_test.values  # 검증용 실제 수익률
        executed_trades = []      # 수익률 저장
        trade_dates = []          # (옵션) 체결 시간 인덱스 저장

        next_trade_idx = 0

        for i in range(len(pred_probs)):
            # 1) 쿨다운 기간 체크
            if i < next_trade_idx:
                continue

            # 2) 진입 조건: 확률 > Threshold
            if pred_probs[i] >= target_threshold:
                # 진입 (실제 수익률 기록)
                trade_ret = returns_arr[i]
                executed_trades.append(trade_ret)

                # 진입했으면 일정 시간(cooldown) 동안은 재진입 금지
                # 30분(vertical barrier)을 다 기다리지 않고, 5분 뒤부터 다시 기회 탐색
                next_trade_idx = i + cooldown

        # ---------------------------------------------------------
        # [개선 4] 결과 집계 및 출력
        # ---------------------------------------------------------
        if not executed_trades:
            print("\n>>> [Warning] 조정 후에도 진입 신호가 없습니다. 모델 학습이 제대로 되지 않았거나 피처를 점검하세요.")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # 승률 계산
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades if n_trades > 0 else 0

        # 수익률 통계
        avg_return = np.mean(executed_trades)
        cum_return = np.prod(executed_trades + 1) - 1
        std_return = np.std(executed_trades)

        # 샤프 비율 (단순화: 무위험이자율 0 가정)
        sharpe_ratio = (avg_return / std_return) * np.sqrt(n_trades) if std_return != 0 else 0

        # MDD (Maximum Drawdown) 계산
        cumulative_returns = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cumulative_returns)
        drawdown = (cumulative_returns - peak) / peak
        max_drawdown = drawdown.min()

        print("-" * 40)
        print(f" [최종 백테스트 결과 (N={n_trades})]")
        print("-" * 40)
        print(f" 1. 승률 (Win Rate)      : {win_rate * 100:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_return * 100:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_return * 100:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe_ratio:.4f}")
        print(f" 5. MDD (Max Drawdown)   : {max_drawdown * 100:.2f}%")

        # 손익비 출력
        avg_win = win_trades.mean() if len(win_trades) > 0 else 0
        avg_loss = loss_trades.mean() if len(loss_trades) > 0 else 0
        if avg_loss != 0:
            pnl_ratio = abs(avg_win / avg_loss)
            print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        else:
            print(f" 6. 손익비 (P/L Ratio)    : Inf")
        print("-" * 40)
# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    import joblib
    from datetime import datetime

    bot = BitcoinTradingModel(CONFIG)

    bot.load_data()
    bot.apply_labeling()
    bot.split_and_scale()
    bot.run_optuna()
    cls_model = bot.train_final_models()

    # 모델 저장
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_filename = f"btc_trading_model_{timestamp}.pkl"

    joblib.dump(cls_model, model_filename)
    print(f"\n>>> 모델 저장 완료: {model_filename}")

    # 선택사항: 최고 성능 파라미터도 함께 저장
    params_filename = f"best_params_{timestamp}.pkl"
    joblib.dump(bot.best_params_class, params_filename)
    print(f">>> 하이퍼파라미터 저장 완료: {params_filename}")

    print("\n>>> 프로세스 완료")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (classification_report, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 하이퍼파라미터 및 경로 설정
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # [New] 거래 비용 설정 (편도 기준)
    'fee_rate': 0.00025,      # 0.025% 업비트 수수료
    'slippage': 0.00020,      # 0.020% (슬리피지 가정)

    'triple_barrier': {
        'span': 100,          # 변동성 계산 기간
        'pt': 1.5,            # 익절 배수
        'sl': 1.5,            # 손절 배수
        'vertical_barrier': 30 # 최대 보유 기간 (캔들 수)
    },
    'optuna_trials': 5,
    'purge_gap': 10,
    'meta_labeling_threshold': 0.65
}


class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None

        # 데이터셋
        self.X_train = None
        self.X_test = None
        self.y_class_train = None  # 분류 타겟 (0, 1)
        self.y_class_test = None

        # 검증용 수익률 데이터 (학습엔 안 씀)
        self.y_return_test = None

        # 모델
        self.classifier = None

        self.feature_cols = []
        self.best_params_class = {}

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")

        # [수정] 더미 데이터 생성 로직 삭제
        if not pd.io.common.file_exists(self.config['file_path']):
             raise FileNotFoundError(f"파일을 찾을 수 없습니다: {self.config['file_path']}")

        self.df = pd.read_csv(self.config['file_path'])
        self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        self.df = self.df.set_index('datetime').sort_index()
        self.df.dropna(inplace=True)
        print(f"    >>> 데이터 로드 완료: {self.df.shape}")


    # =============================================================================
    # 2. Triple Barrier (분류 라벨 생성)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        print("[2] Triple Barrier 레이블링 (Logic Improved) 적용 중...")

        self.df['volatility'] = self._get_volatility(self.df['close'], span=self.config['triple_barrier']['span'])

        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        # 비용 설정 로드
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00020)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # 최소 요구 수익률 (비용의 1.5배 + 마진)
        MIN_RET = round_trip_cost * 1.5 + 0.0005

        labels = []  # 0: 실패/손절/횡보, 1: 성공/익절
        returns = [] # 실제 수익률 (검증용)

        closes = self.df['close'].values
        vols = self.df['volatility'].values
        highs = self.df['high'].values
        lows = self.df['low'].values

        n_samples = len(closes)

        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.001)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = np.nan
            ret = np.nan
            touched = False

            for j in range(1, t_final + 1):
                future_close = closes[i + j]
                if future_close >= upper:
                    # [수정] 최소 수익률 체크
                    temp_ret = (upper - current_price) / current_price
                    if temp_ret > MIN_RET:
                        label = 1
                    else:
                        label = 0
                    ret = temp_ret
                    touched = True
                    break
                if future_close <= lower:
                    label = 0
                    ret = (lower - current_price) / current_price
                    touched = True
                    break

            if not touched:
                # Vertical Barrier (시간 초과)
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price

                # [수정] 최소 수익률 체크
                if ret > MIN_RET:
                    label = 1
                else:
                    label = 0

            labels.append(label)
            returns.append(ret)

        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns # 검증용

        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화 (Rolling Scaling)
    # =============================================================================
    def split_and_scale(self):
        print("[3] 데이터 분할 및 스케일링 (Rolling Scaling)...")

        exclude_cols = ['target_class', 'target_return', 'open', 'high', 'low', 'close', 'volume', 'value', 'volatility', 'datetime']
        self.feature_cols = [col for col in self.df.columns if col not in exclude_cols]

        if not self.feature_cols:
            self.feature_cols = ['open', 'high', 'low', 'volume']

        X = self.df[self.feature_cols]

        window = 1000
        rolling_mean = X.rolling(window=window).mean()
        rolling_std = X.rolling(window=window).std()
        rolling_std = rolling_std.replace(0, 1)

        X_scaled = (X - rolling_mean) / rolling_std

        valid_idx = X_scaled.dropna().index
        X_scaled = X_scaled.loc[valid_idx]
        y_c = self.df.loc[valid_idx, 'target_class']
        y_ret = self.df.loc[valid_idx, 'target_return']

        split_point = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_point]
        self.X_test = X_scaled.iloc[split_point:]

        self.y_class_train = y_c.iloc[:split_point]
        self.y_class_test = y_c.iloc[split_point:]

        # 검증용 수익률 (Test set만 필요)
        self.y_return_test = y_ret.iloc[split_point:]

        # 클리핑
        self.X_train = self.X_train.clip(-5, 5)
        self.X_test = self.X_test.clip(-5, 5)

        print(f"    >>> 학습 셋: {self.X_train.shape}")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    def _purged_cv_score(self, model, X, y, n_splits=3):
        kfold = KFold(n_splits=n_splits, shuffle=False)
        scores = []
        gap = self.config['purge_gap']

        for train_idx, val_idx in kfold.split(X):
            val_start = val_idx[0]
            val_end = val_idx[-1]

            real_train_idx = [i for i in train_idx if i < val_start - gap or i > val_end + gap]

            if len(real_train_idx) < 100: continue

            X_tr = X.iloc[real_train_idx]
            y_tr = y.iloc[real_train_idx]
            X_val = X.iloc[val_idx]
            y_val = y.iloc[val_idx]

            model.fit(X_tr, y_tr, verbose=False)
            pred = model.predict(X_val)

            scores.append(f1_score(y_val, pred, average='macro'))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝 (Classification Only)
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        # 불균형 비율 계산
        neg = (self.y_class_train == 0).sum()
        pos = (self.y_class_train == 1).sum()
        imbalance_ratio = neg / pos if pos > 0 else 1.0

        print(f"    >>> 데이터 불균형 비율(Neg/Pos): {imbalance_ratio:.2f}")
        print("    >>> [수정] scale_pos_weight 고정값 제거 -> Optuna 튜닝 파라미터로 전환")

        def objective_cls(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'gamma': trial.suggest_float('gamma', 0, 5),

                # [요구사항 반영] L2 정규화 (reg_lambda) 적용 및 튜닝
                'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 5.0),

                # scale_pos_weight 튜닝
                'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, imbalance_ratio + 2.0),
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }
            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])
        self.best_params_class = study_cls.best_params

        print(f"    >>> Best Params: {self.best_params_class}")
        print("    >>> 튜닝 완료.")

    # =============================================================================
    # 6. 최종 모델 학습 및 Meta Labeling 검증
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습 및 Meta Labeling 검증...")

        self.classifier = xgb.XGBClassifier(**self.best_params_class, n_jobs=-1)
        self.classifier.fit(self.X_train, self.y_class_train)

        self._evaluate()

        return self.classifier

    def _evaluate(self):
        """
        [수정된 평가 로직]
        1. 예측 확률 분포 진단 (Diagnosis) 기능 복원
        2. 동적 Threshold 제거 -> Config 고정값(0.60) 사용
        3. Cooldown (5 캔들) 적용하여 백테스팅
        """
        # 1. 모델 예측 (확률값 추출)
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]
        probs_series = pd.Series(pred_probs)

        print("\n" + "="*60)
        print(" [ Meta Labeling 전략 평가 (Fixed Threshold) ]")
        print("="*60)

        # ---------------------------------------------------------
        # [Diagnosis] 예측 확률 분포 진단 (복원된 부분)
        # ---------------------------------------------------------
        print("\n[Diagnosis] 예측 확률 분포 요약:")
        # 주요 분위수(Percentiles)를 출력하여 모델의 확신도 분포를 확인
        print(probs_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

        # ---------------------------------------------------------
        # 비용 설정
        # ---------------------------------------------------------
        fee_one_way = self.config.get('fee_rate', 0.00025)
        slip_one_way = self.config.get('slippage', 0.00020)
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        print(f"\n>>> 적용 왕복 비용: {round_trip_cost*100:.3f}%")

        # [요구사항 반영] 동적 Threshold 로직 삭제 -> 고정값 사용
        target_threshold = self.config['meta_labeling_threshold']
        print(f">>> 고정 Threshold 적용: {target_threshold}")

        # ---------------------------------------------------------
        # 백테스팅 로직 (Cooldown 적용)
        # ---------------------------------------------------------
        # [요구사항] Cooldown 5분(캔들) 고정
        cooldown = 5
        print(f">>> 진입 후 쿨다운(대기): {cooldown} 캔들")

        returns_arr = self.y_return_test.values  # 검증용 실제 수익률
        executed_trades = []      # 수익률 저장

        next_trade_idx = 0

        for i in range(len(pred_probs)):
            # 1) 쿨다운 기간 체크
            if i < next_trade_idx:
                continue

            # 2) 진입 조건: 확률 > Threshold
            if pred_probs[i] >= target_threshold:
                # 수익률 계산 (순수 수익률 - 왕복 비용)
                raw_return = returns_arr[i]
                net_return = raw_return - round_trip_cost

                executed_trades.append(net_return)

                # 진입했으면 일정 시간(cooldown) 동안은 재진입 금지
                next_trade_idx = i + cooldown

        # ---------------------------------------------------------
        # 결과 집계 및 출력
        # ---------------------------------------------------------
        if not executed_trades:
            print("\n>>> [Warning] 조건에 맞는 진입 신호가 없습니다. (No Trades)")
            # 진입이 없어도 진단 결과는 보고 싶으므로 여기서 리턴하지만,
            # 로그는 남겨줍니다.
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # 승률 계산
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades if n_trades > 0 else 0

        # 수익률 통계
        avg_return = np.mean(executed_trades)
        cum_return = np.prod(executed_trades + 1) - 1
        std_return = np.std(executed_trades)

        # 샤프 비율 (단순화: 무위험이자율 0 가정)
        sharpe_ratio = (avg_return / std_return) * np.sqrt(n_trades) if std_return != 0 else 0

        # MDD (Maximum Drawdown) 계산
        cumulative_returns = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cumulative_returns)
        drawdown = (cumulative_returns - peak) / peak
        max_drawdown = drawdown.min()

        # 손익비 출력
        avg_win = win_trades.mean() if len(win_trades) > 0 else 0
        avg_loss = loss_trades.mean() if len(loss_trades) > 0 else 0

        if avg_loss != 0:
            pnl_ratio = abs(avg_win / avg_loss)
        else:
            pnl_ratio = float('inf')

        print("-" * 40)
        print(f" [최종 백테스트 결과 (N={n_trades})]")
        print("-" * 40)
        print(f" 1. 승률 (Win Rate)       : {win_rate * 100:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_return * 100:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_return * 100:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe_ratio:.4f}")
        print(f" 5. MDD (Max Drawdown)   : {max_drawdown * 100:.2f}%")
        print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        print("-" * 40)


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)

    bot.load_data()
    bot.apply_labeling()
    bot.split_and_scale()
    bot.run_optuna()
    cls_model = bot.train_final_models()

    # 모델 저장
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_filename = f"btc_trading_model_{timestamp}.pkl"

    joblib.dump(cls_model, model_filename)
    print(f"\n>>> 모델 저장 완료: {model_filename}")

    # 선택사항: 최고 성능 파라미터도 함께 저장
    params_filename = f"best_params_{timestamp}.pkl"
    joblib.dump(bot.best_params_class, params_filename)
    print(f">>> 하이퍼파라미터 저장 완료: {params_filename}")

    print("\n>>> 프로세스 완료")


[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (516465, 92)
[2] Triple Barrier 레이블링 (Logic Improved) 적용 중...
    >>> 레이블링 완료. 상승(1) 비율: 1.95%
[3] 데이터 분할 및 스케일링 (Rolling Scaling)...


[I 2025-12-02 06:21:07,913] A new study created in memory with name: no-name-204535b2-41bf-454c-a308-40c2edb809ea


    >>> 학습 셋: (412348, 85)
[4] 하이퍼파라미터 튜닝 (Trials: 5)
    >>> 데이터 불균형 비율(Neg/Pos): 41.37
    >>> [수정] scale_pos_weight 고정값 제거 -> Optuna 튜닝 파라미터로 전환


[I 2025-12-02 06:21:22,643] Trial 0 finished with value: 0.5343138720390048 and parameters: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.039706016289501024, 'subsample': 0.7096905536430287, 'colsample_bytree': 0.6279093688538189, 'gamma': 3.2383593385451546, 'reg_lambda': 2.731010661332487, 'scale_pos_weight': 19.16618955982096}. Best is trial 0 with value: 0.5343138720390048.
[I 2025-12-02 06:21:40,647] Trial 1 finished with value: 0.5352561847067304 and parameters: {'n_estimators': 438, 'max_depth': 3, 'learning_rate': 0.07674251376572261, 'subsample': 0.6860579574065107, 'colsample_bytree': 0.6630921215065787, 'gamma': 1.5040653475077626, 'reg_lambda': 2.2216230909732695, 'scale_pos_weight': 29.830626904141308}. Best is trial 1 with value: 0.5352561847067304.
[I 2025-12-02 06:22:03,605] Trial 2 finished with value: 0.5064527693200785 and parameters: {'n_estimators': 373, 'max_depth': 8, 'learning_rate': 0.0975460341601213, 'subsample': 0.8360237915022763, 'colsample_bytr

    >>> Best Params: {'n_estimators': 438, 'max_depth': 3, 'learning_rate': 0.07674251376572261, 'subsample': 0.6860579574065107, 'colsample_bytree': 0.6630921215065787, 'gamma': 1.5040653475077626, 'reg_lambda': 2.2216230909732695, 'scale_pos_weight': 29.830626904141308}
    >>> 튜닝 완료.
[5] 최종 모델 학습 및 Meta Labeling 검증...

 [ Meta Labeling 전략 평가 (Fixed Threshold) ]

[Diagnosis] 예측 확률 분포 요약:
count    1.030880e+05
mean     1.344654e-01
std      2.168840e-01
min      1.275440e-07
50%      1.765568e-02
75%      1.771987e-01
90%      4.947527e-01
95%      6.645410e-01
99%      8.481523e-01
max      9.814093e-01
dtype: float64

>>> 적용 왕복 비용: 0.090%
>>> 고정 Threshold 적용: 0.65
>>> 진입 후 쿨다운(대기): 5 캔들
----------------------------------------
 [최종 백테스트 결과 (N=1360)]
----------------------------------------
 1. 승률 (Win Rate)       : 43.90%
 2. 평균 수익률 (Avg Ret) : -0.0904%
 3. 누적 수익률 (Cum Ret) : -70.83%
 4. 샤프 지수 (Sharpe)    : -20.1466
 5. MDD (Max Drawdown)   : -70.96%
 6. 손익비 (P/L Ratio)    : 0.34
--

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, precision_recall_curve)
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 24시간(1440분) 스윙 트레이딩 최적화 설정
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # [Cost] 거래 비용 설정
    'fee_rate': 0.00025,      # 0.025% (업비트/바이낸스 평균)
    'slippage': 0.00020,      # 0.020% (긴 호흡이므로 슬리피지 영향 상대적 감소)

    # [Strategy] 24시간 보유 전략 파라미터
    'triple_barrier': {
        'span': 1440,             # [수정] 변동성 계산 기간: 1일(1440분) 기준
        'pt': 2.0,                # [유지] 익절: 변동성의 2배 (큰 추세 추종)
        'sl': 1.0,                # [유지] 손절: 변동성의 1배 (방어적)
        'vertical_barrier': 1440  # [수정] 최대 보유 기간: 24시간 (1440분)
    },

    # [Feature] 데이터 윈도우 설정
    'feature_windows': [1440],    # [수정] 24시간 기준의 상대적 위치 파악

    'optuna_trials': 10,          # [권장] 탐색 횟수 소폭 상향
    'purge_gap': 1500,            # [중요] 보유기간(1440)보다 커야 데이터 누수 방지 가능
    'meta_labeling_threshold': 0.6
}

class FocalLossObjective:
    def __init__(self, alpha, gamma):
        self.alpha = alpha
        self.gamma = gamma

    def get_objective(self, y_true, y_pred):
        labels = y_true
        preds = y_pred
        preds = 1.0 / (1.0 + np.exp(-preds))
        preds = np.clip(preds, 1e-7, 1.0 - 1e-7)
        pt = np.where(labels == 1, preds, 1 - preds)
        alpha_t = np.where(labels == 1, self.alpha, 1 - self.alpha)
        grad = alpha_t * (1 - pt)**self.gamma * (preds - labels)
        hess = alpha_t * (1 - pt)**self.gamma * preds * (1 - preds) * \
               (1 + self.gamma * (1 - pt) * np.log(pt))
        return grad, np.maximum(hess, 1e-6)

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_class_train = None
        self.y_class_test = None
        self.y_return_test = None
        self.classifier = None
        self.feature_cols = []
        self.best_params_class = {}

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        if not pd.io.common.file_exists(self.config['file_path']):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {self.config['file_path']}")

        self.df = pd.read_csv(self.config['file_path'])
        self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        self.df = self.df.set_index('datetime').sort_index()

        # [보수적 수정] 결측치가 너무 많은 앞부분 제거
        self.df.dropna(inplace=True)
        print(f"    >>> 데이터 로드 완료: {self.df.shape}")

    # =============================================================================
    # 2. Triple Barrier (24h 최적화)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        # 일간 수익률이 아닌, 캔들 간 수익률의 EWMA 변동성
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        print("[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...")

        # 1) 변동성 계산 (기간: 1440분)
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        fee_one_way = self.config['fee_rate']
        slip_one_way = self.config['slippage']
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # [수정] 긴 호흡이므로 목표 수익률을 조금 더 여유 있게 잡음
        multiplier = 1.2
        extra_margin = 0.001    # 0.1% 추가 마진
        MIN_RET = round_trip_cost * multiplier + extra_margin

        print(f"    >>> Holding Period: {t_final} mins (24h)")
        print(f"    >>> MIN_RET (Target): {MIN_RET*100:.3f}%")

        labels = []
        returns = []
        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values
        n_samples = len(closes)

        # 벡터 연산으로 최적화 가능하나, 로직 명확성을 위해 기존 Loop 유지 (보수적 접근)
        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.002) # [수정] 24시간 변동성 최소값 보정 (너무 작으면 안됨)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = 0
            ret = 0.0
            touched = False

            # Horizon Loop
            for j in range(1, t_final + 1):
                # Take Profit
                if highs[i + j] >= upper:
                    ret = (upper - current_price) / current_price
                    if ret > MIN_RET: label = 1
                    touched = True
                    break
                # Stop Loss
                if lows[i + j] <= lower:
                    ret = (lower - current_price) / current_price
                    label = 0
                    touched = True
                    break

            # Time Limit (Vertical Barrier)
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price
                if ret > MIN_RET: label = 1
                else: label = 0

            labels.append(label)
            returns.append(ret)

        # Padding for the end
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns
        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화 (Multi-Window Scaling)
    # =============================================================================
    def split_and_scale(self):
        """
        [수정] 24시간 예측을 위해 장기 Window를 기준으로 Scaling 수행
        """
        print("[3] 데이터 분할 및 스케일링 (24h Optimized)...")
        exclude = ['target_class', 'target_return', 'open', 'high', 'low', 'close', 'volume', 'datetime', 'volatility']
        base_features = [c for c in self.df.columns if c not in exclude]

        # [핵심 수정] 단일 200 윈도우 -> 1440(1일) 윈도우 사용
        # 이유: 24시간 뒤를 예측하려면, 현재 데이터가 지난 24시간 대비 어디에 위치하는지(Z-score)가 가장 중요함.
        main_window = self.config['feature_windows'][0] # 1440

        X_df = pd.DataFrame(index=self.df.index)

        # 기존 피처들을 24시간 기준 Rolling Z-Score로 변환
        # (값이 클수록 지난 24시간 평균 대비 고평가, 작으면 저평가)
        for col in base_features:
            roll_mean = self.df[col].rolling(window=main_window).mean()
            roll_std = self.df[col].rolling(window=main_window).std().replace(0, 1)

            # Z-Score Normalization
            X_df[col] = (self.df[col] - roll_mean) / roll_std

        # NaN 제거 (1440분 데이터 확보 필요)
        X_df.dropna(inplace=True)

        # 인덱스 동기화
        common_idx = X_df.index.intersection(self.df.index)
        X_scaled = X_df.loc[common_idx]

        # Train/Test Split
        split_idx = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_idx].clip(-5, 5)
        self.X_test = X_scaled.iloc[split_idx:].clip(-5, 5)

        self.y_class_train = self.df.loc[X_scaled.index, 'target_class'].iloc[:split_idx]
        self.y_class_test = self.df.loc[X_scaled.index, 'target_class'].iloc[split_idx:]
        self.y_return_test = self.df.loc[X_scaled.index, 'target_return'].iloc[split_idx:]

        print(f"    >>> 학습 셋: {self.X_train.shape} (Window: {main_window})")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    def _purged_cv_score(self, model, X, y):
        kf = KFold(n_splits=3, shuffle=False)
        scores = []
        gap = self.config['purge_gap']  # 1500 (24h 이상)

        for tr_idx, val_idx in kf.split(X):
            # [중요] Validation 시작점보다 'gap'만큼 이전에 끝나는 데이터만 학습에 사용
            # 24시간 보유 전략이므로, 최소 24시간 전 데이터까지만 봐야 함.
            tr_idx = tr_idx[tr_idx < val_idx[0] - gap]

            if len(tr_idx) < 1000: continue # 데이터 부족 시 스킵

            model.fit(X.iloc[tr_idx], y.iloc[tr_idx], verbose=False)
            pred = model.predict(X.iloc[val_idx])
            scores.append(accuracy_score(y.iloc[val_idx], pred))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        def objective_cls(trial):
            fl_alpha = trial.suggest_float('fl_alpha', 0.60, 0.90) # 범위 소폭 확대
            fl_gamma = trial.suggest_float('fl_gamma', 1.0, 5.0)
            fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)

            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 1000), # 긴 호흡 데이터라 복잡도 증가 예상
                'max_depth': trial.suggest_int('max_depth', 5, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 0.1, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'gamma': trial.suggest_float('xgb_gamma', 0.0, 3.0),

                'objective': fl_obj.get_objective,
                'eval_metric': 'logloss',
                'disable_default_eval_metric': 1,
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])

        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")
        return study_cls

    # =============================================================================
    # 6. 최종 모델 학습
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습...")
        bp = self.best_params_class.copy()
        fl_alpha = bp.pop('fl_alpha')
        fl_gamma = bp.pop('fl_gamma')

        if 'xgb_gamma' in bp: bp['gamma'] = bp.pop('xgb_gamma')

        fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)
        self.classifier = xgb.XGBClassifier(
            **bp,
            objective=fl_obj.get_objective,
            scale_pos_weight=1.0,
            n_jobs=-1
        )
        self.classifier.fit(self.X_train, self.y_class_train)
        print(f"    >>> 모델 학습 완료.")
        return self.classifier

    def _find_optimal_threshold(self, pred_probs, y_true):
        precision, recall, thresholds = precision_recall_curve(y_true, pred_probs)
        numerator = 2 * precision * recall
        denominator = precision + recall
        f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)
        best_idx = np.argmax(f1_scores)
        optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        max_f1 = f1_scores[best_idx]
        return optimal_threshold, max_f1

    # =============================================================================
    # 7. 평가 (Evaluate)
    # =============================================================================
    def evaluate(self):
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]
        probs_series = pd.Series(pred_probs)

        print("\n" + "="*60)
        print(" [ 24시간 스윙 전략 평가 (1440 Ticks) ]")
        print("="*60)

        # 비용 계산
        fee = self.config['fee_rate']
        slip = self.config['slippage']
        cost = 2 * (fee + slip)

        # Threshold 최적화
        opt_th, max_f1 = self._find_optimal_threshold(pred_probs, self.y_class_test)

        # [수정] 24시간 전략은 진입 기회가 적으므로, 너무 엄격한 필터링보다는
        # F1 Score 최적값을 신뢰하되, 최소 0.5(중립) 이상은 넘도록 설정
        final_th = max(opt_th, 0.5)

        # 안전장치: 상위 30% 이내 확률일 때만 진입 (너무 잦은 매매 방지)
        safety_th = probs_series.quantile(0.70)
        if final_th < safety_th:
            final_th = safety_th

        print(f"    >>> 적용 Threshold: {final_th:.4f} (Max F1: {max_f1:.4f})")

        # 백테스팅
        returns_arr = self.y_return_test.values
        executed_trades = []

        # [수정] 쿨다운 기간: 24시간 (1440분)
        # 한 포지션을 잡으면 24시간 동안은 추가 진입 금지 (단일 포지션 가정)
        cooldown = self.config['triple_barrier']['vertical_barrier']
        next_trade_idx = 0

        for i in range(len(pred_probs)):
            if i < next_trade_idx: continue

            if pred_probs[i] >= final_th:
                raw_return = returns_arr[i]
                net_return = raw_return - cost
                executed_trades.append(net_return)
                next_trade_idx = i + cooldown

        if not executed_trades:
            print(">>> [Warning] 진입 신호 없음.")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)
        win_rate = len(executed_trades[executed_trades > 0]) / n_trades if n_trades > 0 else 0
        avg_ret = np.mean(executed_trades)
        cum_ret = np.prod(executed_trades + 1) - 1

        # 연율화 (단순 계산: 데이터 기간에 따라 다름, 여기선 단순 누적)
        print("-" * 40)
        print(f" 총 거래 횟수: {n_trades} 회")
        print(f" 승률 (Win Rate): {win_rate * 100:.2f}%")
        print(f" 평균 수익률: {avg_ret * 100:.4f}%")
        print(f" 누적 수익률: {cum_ret * 100:.2f}%")
        print("-" * 40)

if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()
    bot.apply_labeling()
    bot.split_and_scale()
    bot.run_optuna()
    model = bot.train_final_models()
    bot.evaluate()

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (475425, 128)
[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...
    >>> Holding Period: 1440 mins (24h)
    >>> MIN_RET (Target): 0.208%
    >>> 레이블링 완료. 상승(1) 비율: 32.78%
[3] 데이터 분할 및 스케일링 (24h Optimized)...


[I 2025-12-02 12:48:37,277] A new study created in memory with name: no-name-4a6d8b2e-1850-4bf4-b432-13afc214f9f6


    >>> 학습 셋: (378036, 122) (Window: 1440)
[4] 하이퍼파라미터 튜닝 (Trials: 10)


[I 2025-12-02 12:49:10,480] Trial 0 finished with value: 0.6736620321874107 and parameters: {'fl_alpha': 0.8559511243631053, 'fl_gamma': 4.467541185279039, 'n_estimators': 605, 'max_depth': 6, 'learning_rate': 0.04610303559528496, 'subsample': 0.7701143060805286, 'colsample_bytree': 0.7563333861068806, 'reg_alpha': 0.04899417184562679, 'reg_lambda': 0.11525481069882668, 'xgb_gamma': 2.0241619807544766}. Best is trial 0 with value: 0.6736620321874107.
[I 2025-12-02 12:49:44,914] Trial 1 finished with value: 0.6736620321874107 and parameters: {'fl_alpha': 0.6353047357506468, 'fl_gamma': 3.414808975433159, 'n_estimators': 584, 'max_depth': 7, 'learning_rate': 0.006352994189417088, 'subsample': 0.7460811664961222, 'colsample_bytree': 0.8929907058752814, 'reg_alpha': 0.0015038947252700198, 'reg_lambda': 0.00485805416712093, 'xgb_gamma': 0.7990630272210905}. Best is trial 0 with value: 0.6736620321874107.
[I 2025-12-02 12:50:20,622] Trial 2 finished with value: 0.6736620321874107 and paramet

    >>> Best Params: {'fl_alpha': 0.8559511243631053, 'fl_gamma': 4.467541185279039, 'n_estimators': 605, 'max_depth': 6, 'learning_rate': 0.04610303559528496, 'subsample': 0.7701143060805286, 'colsample_bytree': 0.7563333861068806, 'reg_alpha': 0.04899417184562679, 'reg_lambda': 0.11525481069882668, 'xgb_gamma': 2.0241619807544766}
[5] 최종 모델 학습...
    >>> 모델 학습 완료.

 [ 24시간 스윙 전략 평가 (1440 Ticks) ]
    >>> 적용 Threshold: 0.5000 (Max F1: 0.4924)
----------------------------------------
 총 거래 횟수: 66 회
 승률 (Win Rate): 40.91%
 평균 수익률: -0.0445%
 누적 수익률: -2.93%
----------------------------------------


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, precision_recall_curve)
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 24시간(1440분) 스윙 트레이딩 최적화 설정
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # [Cost] 거래 비용 설정
    'fee_rate': 0.00025,      # 0.025% (업비트/바이낸스 평균)
    'slippage': 0.00020,      # 0.020% (긴 호흡이므로 슬리피지 영향 상대적 감소)

    # [Strategy] 24시간 보유 전략 파라미터
    'triple_barrier': {
        'span': 1440,             # [수정] 변동성 계산 기간: 1일(1440분) 기준
        'pt': 2.0,                # [유지] 익절: 변동성의 2배 (큰 추세 추종)
        'sl': 1.0,                # [유지] 손절: 변동성의 1배 (방어적)
        'vertical_barrier': 1440  # [수정] 최대 보유 기간: 24시간 (1440분)
    },

    # [Feature] 데이터 윈도우 설정
    'feature_windows': [1440],    # [수정] 24시간 기준의 상대적 위치 파악

    'optuna_trials': 10,          # [권장] 탐색 횟수 소폭 상향
    'purge_gap': 1500,            # [중요] 보유기간(1440)보다 커야 데이터 누수 방지 가능
    'meta_labeling_threshold': 0.6
}

class FocalLossObjective:
    def __init__(self, alpha, gamma):
        self.alpha = alpha
        self.gamma = gamma

    def get_objective(self, y_true, y_pred):
        labels = y_true
        preds = y_pred
        preds = 1.0 / (1.0 + np.exp(-preds))
        preds = np.clip(preds, 1e-7, 1.0 - 1e-7)
        pt = np.where(labels == 1, preds, 1 - preds)
        alpha_t = np.where(labels == 1, self.alpha, 1 - self.alpha)
        grad = alpha_t * (1 - pt)**self.gamma * (preds - labels)
        hess = alpha_t * (1 - pt)**self.gamma * preds * (1 - preds) * \
               (1 + self.gamma * (1 - pt) * np.log(pt))
        return grad, np.maximum(hess, 1e-6)

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_class_train = None
        self.y_class_test = None
        self.y_return_test = None
        self.classifier = None
        self.feature_cols = []
        self.best_params_class = {}
        # [수정] 레이블 존재 여부 확인 플래그 추가
        self.is_labeled = False

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        if not pd.io.common.file_exists(self.config['file_path']):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {self.config['file_path']}")

        self.df = pd.read_csv(self.config['file_path'])
        self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        self.df = self.df.set_index('datetime').sort_index()

        # [보수적 수정] 결측치가 너무 많은 앞부분 제거
        self.df.dropna(inplace=True)
        print(f"    >>> 데이터 로드 완료: {self.df.shape}")

        # [수정] 레이블 존재 여부 확인
        if 'target_class' in self.df.columns and 'target_return' in self.df.columns:
            self.is_labeled = True
            print("    >>> [Notice] 'target_class'와 'target_return'이 이미 존재합니다. 레이블링 단계는 건너뜁니다.")

        # [수정] labeling이 필요할 경우, 핵심 가격 컬럼 존재 여부 확인
        elif not all(col in self.df.columns for col in ['close', 'high', 'low']):
            raise KeyError(
                "Triple Barrier 레이블링에 필수적인 'close', 'high', 'low' 컬럼이 데이터셋에 없습니다. "
                "레이블링을 건너뛰려면 'target_class'와 'target_return' 컬럼을 데이터셋에 포함하세요."
            )

    # =============================================================================
    # 2. Triple Barrier (24h 최적화)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        # 일간 수익률이 아닌, 캔들 간 수익률의 EWMA 변동성
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        # [수정] 이미 레이블이 있다면 건너뜀
        if self.is_labeled:
            print("[2] 레이블이 이미 존재하여 Triple Barrier 레이블링을 건너뜁니다.")
            return

        print("[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...")

        # 1) 변동성 계산 (기간: 1440분)
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        fee_one_way = self.config['fee_rate']
        slip_one_way = self.config['slippage']
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # [수정] 긴 호흡이므로 목표 수익률을 조금 더 여유 있게 잡음
        multiplier = 1.2
        extra_margin = 0.001    # 0.1% 추가 마진
        MIN_RET = round_trip_cost * multiplier + extra_margin

        print(f"    >>> Holding Period: {t_final} mins (24h)")
        print(f"    >>> MIN_RET (Target): {MIN_RET*100:.3f}%")

        labels = []
        returns = []
        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values
        n_samples = len(closes)

        # 벡터 연산으로 최적화 가능하나, 로직 명확성을 위해 기존 Loop 유지 (보수적 접근)
        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.002) # [수정] 24시간 변동성 최소값 보정 (너무 작으면 안됨)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = 0
            ret = 0.0
            touched = False

            # Horizon Loop
            for j in range(1, t_final + 1):
                # Take Profit
                if highs[i + j] >= upper:
                    ret = (upper - current_price) / current_price
                    if ret > MIN_RET: label = 1
                    touched = True
                    break
                # Stop Loss
                if lows[i + j] <= lower:
                    ret = (lower - current_price) / current_price
                    label = 0
                    touched = True
                    break

            # Time Limit (Vertical Barrier)
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price
                if ret > MIN_RET: label = 1
                else: label = 0

            labels.append(label)
            returns.append(ret)

        # Padding for the end
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns
        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화 (Multi-Window Scaling)
    # =============================================================================
    def split_and_scale(self):
        """
        [수정] 24시간 예측을 위해 장기 Window를 기준으로 Scaling 수행
        """
        print("[3] 데이터 분할 및 스케일링 (24h Optimized)...")
        # [수정] 가격 컬럼이 없을 수도 있으므로, 제외 목록에서 가격 컬럼 체크 로직 제거
        exclude = ['target_class', 'target_return', 'datetime', 'volatility']

        # DataFrame 컬럼 중 exclude에 없는 컬럼을 모두 feature로 사용
        # (open, high, low, close, volume 컬럼이 없다고 가정)
        base_features = [c for c in self.df.columns if c not in exclude]

        # [핵심 수정] 단일 200 윈도우 -> 1440(1일) 윈도우 사용
        # 이유: 24시간 뒤를 예측하려면, 현재 데이터가 지난 24시간 대비 어디에 위치하는지(Z-score)가 가장 중요함.
        main_window = self.config['feature_windows'][0] # 1440

        X_df = pd.DataFrame(index=self.df.index)

        # 기존 피처들을 24시간 기준 Rolling Z-Score로 변환
        # (값이 클수록 지난 24시간 평균 대비 고평가, 작으면 저평가)
        for col in base_features:
            # 롤링 윈도우 계산 시 최소 윈도우 크기를 설정 (데이터 안정성 확보)
            roll_mean = self.df[col].rolling(window=main_window, min_periods=int(main_window * 0.8)).mean()
            roll_std = self.df[col].rolling(window=main_window, min_periods=int(main_window * 0.8)).std().replace(0, 1)

            # Z-Score Normalization
            X_df[col] = (self.df[col] - roll_mean) / roll_std

        # NaN 제거 (1440분 데이터 확보 필요)
        X_df.dropna(inplace=True)

        # 인덱스 동기화
        common_idx = X_df.index.intersection(self.df.index)
        X_scaled = X_df.loc[common_idx]

        # Train/Test Split
        split_idx = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_idx].clip(-5, 5)
        self.X_test = X_scaled.iloc[split_idx:].clip(-5, 5)

        self.y_class_train = self.df.loc[X_scaled.index, 'target_class'].iloc[:split_idx]
        self.y_class_test = self.df.loc[X_scaled.index, 'target_class'].iloc[split_idx:]
        self.y_return_test = self.df.loc[X_scaled.index, 'target_return'].iloc[split_idx:]

        print(f"    >>> 학습 셋: {self.X_train.shape} (Window: {main_window})")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    def _purged_cv_score(self, model, X, y):
        kf = KFold(n_splits=3, shuffle=False)
        scores = []
        gap = self.config['purge_gap']  # 1500 (24h 이상)

        for tr_idx, val_idx in kf.split(X):
            # [중요] Validation 시작점보다 'gap'만큼 이전에 끝나는 데이터만 학습에 사용
            # 24시간 보유 전략이므로, 최소 24시간 전 데이터까지만 봐야 함.
            tr_idx = tr_idx[tr_idx < val_idx[0] - gap]

            if len(tr_idx) < 1000: continue # 데이터 부족 시 스킵

            model.fit(X.iloc[tr_idx], y.iloc[tr_idx], verbose=False)
            pred = model.predict(X.iloc[val_idx])
            scores.append(accuracy_score(y.iloc[val_idx], pred))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        def objective_cls(trial):
            fl_alpha = trial.suggest_float('fl_alpha', 0.60, 0.90) # 범위 소폭 확대
            fl_gamma = trial.suggest_float('fl_gamma', 1.0, 5.0)
            fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)

            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 1000), # 긴 호흡 데이터라 복잡도 증가 예상
                'max_depth': trial.suggest_int('max_depth', 5, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 0.1, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'gamma': trial.suggest_float('xgb_gamma', 0.0, 3.0),

                'objective': fl_obj.get_objective,
                'eval_metric': 'logloss',
                'disable_default_eval_metric': 1,
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])

        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")
        return study_cls

    # =============================================================================
    # 6. 최종 모델 학습
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습...")
        bp = self.best_params_class.copy()
        fl_alpha = bp.pop('fl_alpha')
        fl_gamma = bp.pop('fl_gamma')

        if 'xgb_gamma' in bp: bp['gamma'] = bp.pop('xgb_gamma')

        fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)
        self.classifier = xgb.XGBClassifier(
            **bp,
            objective=fl_obj.get_objective,
            scale_pos_weight=1.0,
            n_jobs=-1
        )
        self.classifier.fit(self.X_train, self.y_class_train)
        print(f"    >>> 모델 학습 완료.")
        return self.classifier

    def _find_optimal_threshold(self, pred_probs, y_true):
        precision, recall, thresholds = precision_recall_curve(y_true, pred_probs)
        numerator = 2 * precision * recall
        denominator = precision + recall
        f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)
        best_idx = np.argmax(f1_scores)
        optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        max_f1 = f1_scores[best_idx]
        return optimal_threshold, max_f1

    # =============================================================================
    # 7. 평가 (Evaluate) - [상세 지표 추가됨]
    # =============================================================================
    def evaluate(self):
        # 1. 모델 예측
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]
        probs_series = pd.Series(pred_probs)

        print("\n" + "="*60)
        print(" [ 24시간 스윙 전략 상세 평가 (Enhanced Metrics) ]")
        print("="*60)

        # ---------------------------------------------------------
        # [Diagnosis] 예측 확률 분포 진단
        # ---------------------------------------------------------
        print("\n[1] 예측 확률 분포 (Confidence Distribution):")
        print(probs_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

        # 비용 설정
        fee = self.config['fee_rate']
        slip = self.config['slippage']
        cost = 2 * (fee + slip)

        # ---------------------------------------------------------
        # [Threshold] 최적화 및 적용
        # ---------------------------------------------------------
        opt_th, max_f1 = self._find_optimal_threshold(pred_probs, self.y_class_test)

        # [수정] 0.5가 너무 낮아 손실이 발생하는 경우를 방지하기 위해
        # F1 최적값과 0.55(약간의 우위) 중 큰 값을 선택하도록 보수적 설정
        final_th = max(opt_th, 0.55)

        # 안전장치: 상위 30% 이내 확률일 때만 진입 (노이즈 필터링)
        safety_th = probs_series.quantile(0.70)
        if final_th < safety_th:
            final_th = safety_th

        print(f"\n[2] 임계값 설정 (Threshold):")
        print(f"    >>> F1 Max Threshold: {opt_th:.4f} (Score: {max_f1:.4f})")
        print(f"    >>> Safety Threshold (Top 30%): {safety_th:.4f}")
        print(f"    >>> 최종 적용 Threshold: {final_th:.4f}")

        # ---------------------------------------------------------
        # [Backtest] 시뮬레이션
        # ---------------------------------------------------------
        returns_arr = self.y_return_test.values
        executed_trades = []

        # [정책] 단일 포지션 사이클 준수: 진입 후 보유기간(24h) 동안 재진입 금지
        # 5분 쿨다운은 다중 포지션을 의미하므로 자금 관리상 위험할 수 있어
        # 보수적으로 vertical_barrier(1440분)를 쿨다운으로 사용합니다.
        cooldown = self.config['triple_barrier']['vertical_barrier']
        next_trade_idx = 0

        for i in range(len(pred_probs)):
            if i < next_trade_idx: continue

            if pred_probs[i] >= final_th:
                # 수익률 계산 (비용 차감)
                raw_return = returns_arr[i]
                net_return = raw_return - cost
                executed_trades.append(net_return)

                # 포지션 점유 처리
                next_trade_idx = i + cooldown

        if not executed_trades:
            print("\n>>> [Warning] 조건에 맞는 진입 신호가 없습니다. (No Trades)")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # ---------------------------------------------------------
        # [Statistics] 상세 성과 지표 계산
        # ---------------------------------------------------------
        # 1. 승률
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades * 100

        # 2. 수익률
        avg_ret = np.mean(executed_trades) * 100
        cum_ret = (np.prod(executed_trades + 1) - 1) * 100
        std_ret = np.std(executed_trades)

        # 3. MDD (Maximum Drawdown)
        cum_returns_series = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cum_returns_series)
        drawdown = (cum_returns_series - peak) / peak
        max_mdd = drawdown.min() * 100

        # 4. Sharpe Ratio (무위험 수익률 0 가정)
        # 연율화: 데이터 기간에 따라 다르지만 여기서는 단순히 거래 횟수 기반 제곱근 사용
        sharpe = (np.mean(executed_trades) / std_ret) * np.sqrt(n_trades) if std_ret > 0 else 0

        # 5. 손익비 (Profit Factor)
        avg_win = win_trades.mean() if len(win_trades) > 0 else 0
        avg_loss = loss_trades.mean() if len(loss_trades) > 0 else 0
        pnl_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else 0

        print("-" * 50)
        print(f" [최종 성과 보고서 (N={n_trades})]")
        print("-" * 50)
        print(f" 1. 승률 (Win Rate)       : {win_rate:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_ret:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_ret:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe:.4f}")
        print(f" 5. MDD (Max Drawdown)    : {max_mdd:.2f}%")
        print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        print("-" * 50)

if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()

    # [수정] 레이블이 이미 존재하면 apply_labeling 건너뛰기
    if not bot.is_labeled:
        bot.apply_labeling()

    bot.split_and_scale()
    bot.run_optuna()
    model = bot.train_final_models()
    bot.evaluate()

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (475425, 128)
[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...
    >>> Holding Period: 1440 mins (24h)
    >>> MIN_RET (Target): 0.208%
    >>> 레이블링 완료. 상승(1) 비율: 32.78%
[3] 데이터 분할 및 스케일링 (24h Optimized)...


[I 2025-12-02 13:06:59,543] A new study created in memory with name: no-name-6c515acb-307e-4c4b-b49c-3c42336a932a


    >>> 학습 셋: (378267, 127) (Window: 1440)
[4] 하이퍼파라미터 튜닝 (Trials: 10)


[I 2025-12-02 13:07:32,306] Trial 0 finished with value: 0.673603565735314 and parameters: {'fl_alpha': 0.897680016303737, 'fl_gamma': 4.49705202277596, 'n_estimators': 594, 'max_depth': 6, 'learning_rate': 0.012032190354966254, 'subsample': 0.7975483392940198, 'colsample_bytree': 0.6223019940081443, 'reg_alpha': 0.007465191816942155, 'reg_lambda': 0.3563920667323218, 'xgb_gamma': 2.7004442731650222}. Best is trial 0 with value: 0.673603565735314.
[I 2025-12-02 13:08:05,476] Trial 1 finished with value: 0.673603565735314 and parameters: {'fl_alpha': 0.8761910815732952, 'fl_gamma': 3.263905900679578, 'n_estimators': 607, 'max_depth': 7, 'learning_rate': 0.014019999556619343, 'subsample': 0.6055078886057499, 'colsample_bytree': 0.8584960689468355, 'reg_alpha': 0.004733094955409709, 'reg_lambda': 0.037501910506580347, 'xgb_gamma': 0.6958527844847208}. Best is trial 0 with value: 0.673603565735314.
[I 2025-12-02 13:08:56,278] Trial 2 finished with value: 0.5828303817145033 and parameters: 

    >>> Best Params: {'fl_alpha': 0.897680016303737, 'fl_gamma': 4.49705202277596, 'n_estimators': 594, 'max_depth': 6, 'learning_rate': 0.012032190354966254, 'subsample': 0.7975483392940198, 'colsample_bytree': 0.6223019940081443, 'reg_alpha': 0.007465191816942155, 'reg_lambda': 0.3563920667323218, 'xgb_gamma': 2.7004442731650222}
[5] 최종 모델 학습...
    >>> 모델 학습 완료.

 [ 24시간 스윙 전략 상세 평가 (Enhanced Metrics) ]

[1] 예측 확률 분포 (Confidence Distribution):
count    94567.0
mean         0.5
std          0.0
min          0.5
50%          0.5
75%          0.5
90%          0.5
95%          0.5
99%          0.5
max          0.5
dtype: float64

[2] 임계값 설정 (Threshold):
    >>> F1 Max Threshold: 0.5000 (Score: 0.4924)
    >>> Safety Threshold (Top 30%): 0.5000
    >>> 최종 적용 Threshold: 0.5500

>>> [Warning] 조건에 맞는 진입 신호가 없습니다. (No Trades)


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, precision_recall_curve)
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 24시간(1440분) 스윙 트레이딩 최적화 설정
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # [Cost] 거래 비용 설정
    'fee_rate': 0.00025,      # 0.025% (업비트/바이낸스 평균)
    'slippage': 0.00020,      # 0.020% (긴 호흡이므로 슬리피지 영향 상대적 감소)

    # [Strategy] 24시간 보유 전략 파라미터
    'triple_barrier': {
        'span': 1440,             # [수정] 변동성 계산 기간: 1일(1440분) 기준
        'pt': 3.0,                # [유지] 익절: 변동성의 2배 (큰 추세 추종)
        'sl': 0.7,                # [유지] 손절: 변동성의 1배 (방어적)
        'vertical_barrier': 1440  # [수정] 최대 보유 기간: 24시간 (1440분)
    },

    # [Feature] 데이터 윈도우 설정
    'feature_windows': [1440],    # [수정] 24시간 기준의 상대적 위치 파악

    'optuna_trials': 50,          # [권장] 탐색 횟수 소폭 상향
    'purge_gap': 1500,            # [중요] 보유기간(1440)보다 커야 데이터 누수 방지 가능
    'meta_labeling_threshold': 0.6
}

class FocalLossObjective:
    def __init__(self, alpha, gamma):
        self.alpha = alpha
        self.gamma = gamma

    def get_objective(self, y_true, y_pred):
        labels = y_true
        preds = y_pred
        # Log-odds -> Probability (Sigmoid)
        preds = 1.0 / (1.0 + np.exp(-preds))
        preds = np.clip(preds, 1e-7, 1.0 - 1e-7)

        pt = np.where(labels == 1, preds, 1 - preds)
        alpha_t = np.where(labels == 1, self.alpha, 1 - self.alpha)

        # Gradient (1차 미분)
        # Standard Logloss Grad: (preds - labels)
        # Focal Loss Grad: alpha_t * (1 - pt)^gamma * (preds - labels)
        grad = alpha_t * (1 - pt)**self.gamma * (preds - labels)

        # Hessian (2차 미분) - [수정됨]
        # 원본 Focal Loss Hessian은 (1 + gamma * ... * log(pt)) 항 때문에 음수가 될 수 있음.
        # XGBoost는 양수 Hessian을 요구하므로, 학습 안정성을 위해
        # "Scaled Logistic Hessian" 근사식을 사용합니다.
        hess = alpha_t * (1 - pt)**self.gamma * preds * (1 - preds)

        return grad, np.maximum(hess, 1e-6)

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_class_train = None
        self.y_class_test = None
        self.y_return_test = None
        self.classifier = None
        self.feature_cols = []
        self.best_params_class = {}
        # [수정] 레이블 존재 여부 확인 플래그 추가
        self.is_labeled = False

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        if not pd.io.common.file_exists(self.config['file_path']):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {self.config['file_path']}")

        self.df = pd.read_csv(self.config['file_path'])
        self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        self.df = self.df.set_index('datetime').sort_index()

        # [보수적 수정] 결측치가 너무 많은 앞부분 제거
        self.df.dropna(inplace=True)
        print(f"    >>> 데이터 로드 완료: {self.df.shape}")

        # [수정] 레이블 존재 여부 확인
        if 'target_class' in self.df.columns and 'target_return' in self.df.columns:
            self.is_labeled = True
            print("    >>> [Notice] 'target_class'와 'target_return'이 이미 존재합니다. 레이블링 단계는 건너뜁니다.")

        # [수정] labeling이 필요할 경우, 핵심 가격 컬럼 존재 여부 확인
        elif not all(col in self.df.columns for col in ['close', 'high', 'low']):
            raise KeyError(
                "Triple Barrier 레이블링에 필수적인 'close', 'high', 'low' 컬럼이 데이터셋에 없습니다. "
                "레이블링을 건너뛰려면 'target_class'와 'target_return' 컬럼을 데이터셋에 포함하세요."
            )

    # =============================================================================
    # 2. Triple Barrier (24h 최적화)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        # 일간 수익률이 아닌, 캔들 간 수익률의 EWMA 변동성
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        # [수정] 이미 레이블이 있다면 건너뜀
        if self.is_labeled:
            print("[2] 레이블이 이미 존재하여 Triple Barrier 레이블링을 건너뜁니다.")
            return

        print("[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...")

        # 1) 변동성 계산 (기간: 1440분)
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        fee_one_way = self.config['fee_rate']
        slip_one_way = self.config['slippage']
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # [수정] 긴 호흡이므로 목표 수익률을 조금 더 여유 있게 잡음
        multiplier = 1.2
        extra_margin = 0.001    # 0.1% 추가 마진
        MIN_RET = round_trip_cost * multiplier + extra_margin

        print(f"    >>> Holding Period: {t_final} mins (24h)")
        print(f"    >>> MIN_RET (Target): {MIN_RET*100:.3f}%")

        labels = []
        returns = []
        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values
        n_samples = len(closes)

        # 벡터 연산으로 최적화 가능하나, 로직 명확성을 위해 기존 Loop 유지 (보수적 접근)
        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.002) # [수정] 24시간 변동성 최소값 보정 (너무 작으면 안됨)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = 0
            ret = 0.0
            touched = False

            # Horizon Loop
            for j in range(1, t_final + 1):
                # Take Profit
                if highs[i + j] >= upper:
                    ret = (upper - current_price) / current_price
                    if ret > MIN_RET: label = 1
                    touched = True
                    break
                # Stop Loss
                if lows[i + j] <= lower:
                    ret = (lower - current_price) / current_price
                    label = 0
                    touched = True
                    break

            # Time Limit (Vertical Barrier)
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price
                if ret > MIN_RET: label = 1
                else: label = 0

            labels.append(label)
            returns.append(ret)

        # Padding for the end
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns
        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화 (Multi-Window Scaling)
    # =============================================================================
    def split_and_scale(self):
        """
        [수정] 24시간 예측을 위해 장기 Window를 기준으로 Scaling 수행
        """
        print("[3] 데이터 분할 및 스케일링 (24h Optimized)...")
        # [수정] 가격 컬럼이 없을 수도 있으므로, 제외 목록에서 가격 컬럼 체크 로직 제거
        exclude = ['target_class', 'target_return', 'datetime', 'volatility']

        # DataFrame 컬럼 중 exclude에 없는 컬럼을 모두 feature로 사용
        # (open, high, low, close, volume 컬럼이 없다고 가정)
        base_features = [c for c in self.df.columns if c not in exclude]

        # [핵심 수정] 단일 200 윈도우 -> 1440(1일) 윈도우 사용
        # 이유: 24시간 뒤를 예측하려면, 현재 데이터가 지난 24시간 대비 어디에 위치하는지(Z-score)가 가장 중요함.
        main_window = self.config['feature_windows'][0] # 1440

        X_df = pd.DataFrame(index=self.df.index)

        # 기존 피처들을 24시간 기준 Rolling Z-Score로 변환
        # (값이 클수록 지난 24시간 평균 대비 고평가, 작으면 저평가)
        for col in base_features:
            # 롤링 윈도우 계산 시 최소 윈도우 크기를 설정 (데이터 안정성 확보)
            roll_mean = self.df[col].rolling(window=main_window, min_periods=int(main_window * 0.8)).mean()
            roll_std = self.df[col].rolling(window=main_window, min_periods=int(main_window * 0.8)).std().replace(0, 1)

            # Z-Score Normalization
            X_df[col] = (self.df[col] - roll_mean) / roll_std

        # NaN 제거 (1440분 데이터 확보 필요)
        X_df.dropna(inplace=True)

        # 인덱스 동기화
        common_idx = X_df.index.intersection(self.df.index)
        X_scaled = X_df.loc[common_idx]

        # Train/Test Split
        split_idx = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_idx].clip(-5, 5)
        self.X_test = X_scaled.iloc[split_idx:].clip(-5, 5)

        self.y_class_train = self.df.loc[X_scaled.index, 'target_class'].iloc[:split_idx]
        self.y_class_test = self.df.loc[X_scaled.index, 'target_class'].iloc[split_idx:]
        self.y_return_test = self.df.loc[X_scaled.index, 'target_return'].iloc[split_idx:]

        print(f"    >>> 학습 셋: {self.X_train.shape} (Window: {main_window})")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    def _purged_cv_score(self, model, X, y):
        kf = KFold(n_splits=3, shuffle=False)
        scores = []
        gap = self.config['purge_gap']  # 1500 (24h 이상)

        for tr_idx, val_idx in kf.split(X):
            # [중요] Validation 시작점보다 'gap'만큼 이전에 끝나는 데이터만 학습에 사용
            # 24시간 보유 전략이므로, 최소 24시간 전 데이터까지만 봐야 함.
            tr_idx = tr_idx[tr_idx < val_idx[0] - gap]

            if len(tr_idx) < 1000: continue # 데이터 부족 시 스킵

            model.fit(X.iloc[tr_idx], y.iloc[tr_idx], verbose=False)

            # [수정] CV 스코어 계산 시에도 Margin -> Proba 변환 명시
            # Scikit-Learn Wrapper는 Custom Objective 사용 시 predict()가 Margin을 반환할 수 있음
            try:
                # 안전한 방법: Margin을 얻어서 Sigmoid
                pred_margin = model.predict(X.iloc[val_idx], output_margin=True)
                pred_proba = 1.0 / (1.0 + np.exp(-pred_margin))
                pred_label = (pred_proba > 0.5).astype(int)
            except TypeError:
                # predict_proba가 정상 작동하는 경우
                pred_label = model.predict(X.iloc[val_idx])

            scores.append(accuracy_score(y.iloc[val_idx], pred_label))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        def objective_cls(trial):
            fl_alpha = trial.suggest_float('fl_alpha', 0.85, 0.95) # 범위 소폭 확대
            fl_gamma = trial.suggest_float('fl_gamma', 0.5, 2.0)
            fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)

            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 1000), # 긴 호흡 데이터라 복잡도 증가 예상
                'max_depth': trial.suggest_int('max_depth', 5, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 0.1, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'gamma': trial.suggest_float('xgb_gamma', 0.0, 3.0),

                'objective': fl_obj.get_objective,
                'eval_metric': 'logloss',
                'disable_default_eval_metric': 1,
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])

        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")
        return study_cls

    # =============================================================================
    # 6. 최종 모델 학습
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습...")
        bp = self.best_params_class.copy()
        fl_alpha = bp.pop('fl_alpha')
        fl_gamma = bp.pop('fl_gamma')

        if 'xgb_gamma' in bp: bp['gamma'] = bp.pop('xgb_gamma')

        fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)
        self.classifier = xgb.XGBClassifier(
            **bp,
            objective=fl_obj.get_objective,
            scale_pos_weight=1.0,
            n_jobs=-1
        )
        self.classifier.fit(self.X_train, self.y_class_train)
        print(f"    >>> 모델 학습 완료.")
        return self.classifier

    def _find_optimal_threshold(self, pred_probs, y_true):
        precision, recall, thresholds = precision_recall_curve(y_true, pred_probs)
        numerator = 2 * precision * recall
        denominator = precision + recall
        f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)
        best_idx = np.argmax(f1_scores)
        optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        max_f1 = f1_scores[best_idx]
        return optimal_threshold, max_f1
    def evaluate(self):
        # 1. 모델 예측
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]
        probs_series = pd.Series(pred_probs)

        print("\n" + "="*60)
        print(" [ 24시간 스윙 전략 상세 평가 (Enhanced Metrics) ]")
        print("="*60)

        # ---------------------------------------------------------
        # [Diagnosis] 예측 확률 분포 진단
        # ---------------------------------------------------------
        print("\n[1] 예측 확률 분포 (Confidence Distribution):")
        print(probs_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

        # 비용 설정
        fee = self.config['fee_rate']
        slip = self.config['slippage']
        cost = 2 * (fee + slip)

        # ---------------------------------------------------------
        # [Threshold] 최적화 및 적용
        # ---------------------------------------------------------
        opt_th, max_f1 = self._find_optimal_threshold(pred_probs, self.y_class_test)

        # [수정] 0.5가 너무 낮아 손실이 발생하는 경우를 방지하기 위해
        # F1 최적값과 0.55(약간의 우위) 중 큰 값을 선택하도록 보수적 설정
        final_th = max(opt_th, 0.55)

        # 안전장치: 상위 30% 이내 확률일 때만 진입 (노이즈 필터링)
        safety_th = probs_series.quantile(0.70)
        if final_th < safety_th:
            final_th = safety_th

        print(f"\n[2] 임계값 설정 (Threshold):")
        print(f"    >>> F1 Max Threshold: {opt_th:.4f} (Score: {max_f1:.4f})")
        print(f"    >>> Safety Threshold (Top 30%): {safety_th:.4f}")
        print(f"    >>> 최종 적용 Threshold: {final_th:.4f}")

        # ---------------------------------------------------------
        # [Backtest] 시뮬레이션
        # ---------------------------------------------------------
        returns_arr = self.y_return_test.values
        executed_trades = []

        # [정책] 단일 포지션 사이클 준수: 진입 후 보유기간(24h) 동안 재진입 금지
        # 5분 쿨다운은 다중 포지션을 의미하므로 자금 관리상 위험할 수 있어
        # 보수적으로 vertical_barrier(1440분)를 쿨다운으로 사용합니다.
        cooldown = self.config['triple_barrier']['vertical_barrier']
        next_trade_idx = 0

        for i in range(len(pred_probs)):
            if i < next_trade_idx: continue

            if pred_probs[i] >= final_th:
                # 수익률 계산 (비용 차감)
                raw_return = returns_arr[i]
                net_return = raw_return - cost
                executed_trades.append(net_return)

                # 포지션 점유 처리
                next_trade_idx = i + cooldown

        if not executed_trades:
            print("\n>>> [Warning] 조건에 맞는 진입 신호가 없습니다. (No Trades)")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # ---------------------------------------------------------
        # [Statistics] 상세 성과 지표 계산
        # ---------------------------------------------------------
        # 1. 승률
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades * 100

        # 2. 수익률
        avg_ret = np.mean(executed_trades) * 100
        cum_ret = (np.prod(executed_trades + 1) - 1) * 100
        std_ret = np.std(executed_trades)

        # 3. MDD (Maximum Drawdown)
        cum_returns_series = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cum_returns_series)
        drawdown = (cum_returns_series - peak) / peak
        max_mdd = drawdown.min() * 100

        # 4. Sharpe Ratio (무위험 수익률 0 가정)
        # 연율화: 데이터 기간에 따라 다르지만 여기서는 단순히 거래 횟수 기반 제곱근 사용
        sharpe = (np.mean(executed_trades) / std_ret) * np.sqrt(n_trades) if std_ret > 0 else 0

        # 5. 손익비 (Profit Factor)
        avg_win = win_trades.mean() if len(win_trades) > 0 else 0
        avg_loss = loss_trades.mean() if len(loss_trades) > 0 else 0
        pnl_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else 0

        print("-" * 50)
        print(f" [최종 성과 보고서 (N={n_trades})]")
        print("-" * 50)
        print(f" 1. 승률 (Win Rate)       : {win_rate:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_ret:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_ret:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe:.4f}")
        print(f" 5. MDD (Max Drawdown)    : {max_mdd:.2f}%")
        print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        print("-" * 50)

if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()

    # [수정] 레이블이 이미 존재하면 apply_labeling 건너뛰기
    if not bot.is_labeled:
        bot.apply_labeling()

    bot.split_and_scale()
    bot.run_optuna()
    model = bot.train_final_models()
    bot.evaluate()

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (475425, 128)
[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...
    >>> Holding Period: 1440 mins (24h)
    >>> MIN_RET (Target): 0.208%
    >>> 레이블링 완료. 상승(1) 비율: 19.20%
[3] 데이터 분할 및 스케일링 (24h Optimized)...


[I 2025-12-02 15:24:38,879] A new study created in memory with name: no-name-3801a4e1-6a80-40ae-8a23-62efcb2f74d5


    >>> 학습 셋: (378267, 127) (Window: 1440)
[4] 하이퍼파라미터 튜닝 (Trials: 50)


[I 2025-12-02 15:25:29,414] Trial 0 finished with value: 0.6388424049679196 and parameters: {'fl_alpha': 0.8870527636606348, 'fl_gamma': 1.2670154898369694, 'n_estimators': 912, 'max_depth': 7, 'learning_rate': 0.007828061950443551, 'subsample': 0.6403755808306869, 'colsample_bytree': 0.7535659651877761, 'reg_alpha': 0.0016017840386277286, 'reg_lambda': 0.5848046586728871, 'xgb_gamma': 1.6362807619900646}. Best is trial 0 with value: 0.6388424049679196.
[I 2025-12-02 15:26:22,731] Trial 1 finished with value: 0.7141423914853794 and parameters: {'fl_alpha': 0.8754728852658977, 'fl_gamma': 1.2282723001834328, 'n_estimators': 880, 'max_depth': 8, 'learning_rate': 0.013261766355925153, 'subsample': 0.8573897265730308, 'colsample_bytree': 0.7163685910172156, 'reg_alpha': 0.0022618504205582845, 'reg_lambda': 0.006685079521739579, 'xgb_gamma': 0.5075243827606836}. Best is trial 1 with value: 0.7141423914853794.
[I 2025-12-02 15:27:01,821] Trial 2 finished with value: 0.6776681550333494 and pa

    >>> Best Params: {'fl_alpha': 0.8698687040746489, 'fl_gamma': 0.51832786633496, 'n_estimators': 797, 'max_depth': 10, 'learning_rate': 0.03644735035077519, 'subsample': 0.874114563223207, 'colsample_bytree': 0.6019529141808603, 'reg_alpha': 0.004718647164971206, 'reg_lambda': 0.005034107609679906, 'xgb_gamma': 0.0030741988524250474}
[5] 최종 모델 학습...
    >>> 모델 학습 완료.

 [ 24시간 스윙 전략 상세 평가 (Enhanced Metrics) ]

[1] 예측 확률 분포 (Confidence Distribution):
count    94567.000000
mean         0.208745
std          0.188382
min          0.000127
50%          0.147538
75%          0.309639
90%          0.489775
95%          0.602617
99%          0.780449
max          0.973258
dtype: float64

[2] 임계값 설정 (Threshold):
    >>> F1 Max Threshold: 0.0045 (Score: 0.3336)
    >>> Safety Threshold (Top 30%): 0.2680
    >>> 최종 적용 Threshold: 0.5500
--------------------------------------------------
 [최종 성과 보고서 (N=56)]
--------------------------------------------------
 1. 승률 (Win Rate)       : 39.29%
 2. 평

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, precision_recall_curve)
import optuna
import warnings
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

# =============================================================================
# [설정] 24시간(1440분) 스윙 트레이딩 최적화 설정
# =============================================================================
CONFIG = {
    'file_path': '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv',
    'test_size': 0.2,
    'random_state': 42,

    # [Cost] 거래 비용 설정
    'fee_rate': 0.00025,      # 0.025% (업비트/바이낸스 평균)
    'slippage': 0.00020,      # 0.020% (긴 호흡이므로 슬리피지 영향 상대적 감소)

    # [Strategy] 24시간 보유 전략 파라미터
    'triple_barrier': {
        'span': 1440,             # [수정] 변동성 계산 기간: 1일(1440분) 기준
        'pt': 3.0,                # [유지] 익절: 변동성의 2배 (큰 추세 추종)
        'sl': 0.7,                # [유지] 손절: 변동성의 1배 (방어적)
        'vertical_barrier': 1440  # [수정] 최대 보유 기간: 24시간 (1440분)
    },

    # [Feature] 데이터 윈도우 설정
    'feature_windows': [1440],    # [수정] 24시간 기준의 상대적 위치 파악

    'optuna_trials': 100,          # Optuna 100번으로 최대 성능 향상
    'purge_gap': 1500,            # [중요] 보유기간(1440)보다 커야 데이터 누수 방지 가능
    'meta_labeling_threshold': 0.6
}

class FocalLossObjective:
    def __init__(self, alpha, gamma):
        self.alpha = alpha
        self.gamma = gamma

    def get_objective(self, y_true, y_pred):
        labels = y_true
        preds = y_pred
        # Log-odds -> Probability (Sigmoid)
        preds = 1.0 / (1.0 + np.exp(-preds))
        preds = np.clip(preds, 1e-7, 1.0 - 1e-7)

        pt = np.where(labels == 1, preds, 1 - preds)
        alpha_t = np.where(labels == 1, self.alpha, 1 - self.alpha)

        # Gradient (1차 미분)
        # Standard Logloss Grad: (preds - labels)
        # Focal Loss Grad: alpha_t * (1 - pt)^gamma * (preds - labels)
        grad = alpha_t * (1 - pt)**self.gamma * (preds - labels)

        # Hessian (2차 미분) - [수정됨]
        # 원본 Focal Loss Hessian은 (1 + gamma * ... * log(pt)) 항 때문에 음수가 될 수 있음.
        # XGBoost는 양수 Hessian을 요구하므로, 학습 안정성을 위해
        # "Scaled Logistic Hessian" 근사식을 사용합니다.
        hess = alpha_t * (1 - pt)**self.gamma * preds * (1 - preds)

        return grad, np.maximum(hess, 1e-6)

class BitcoinTradingModel:
    def __init__(self, config):
        self.config = config
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_class_train = None
        self.y_class_test = None
        self.y_return_test = None
        self.classifier = None
        self.feature_cols = []
        self.best_params_class = {}
        # [수정] 레이블 존재 여부 확인 플래그 추가
        self.is_labeled = False

    # =============================================================================
    # 1. 데이터 로드
    # =============================================================================
    def load_data(self):
        print(f"[1] 데이터 로드 중... ({self.config['file_path']})")
        if not pd.io.common.file_exists(self.config['file_path']):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {self.config['file_path']}")

        self.df = pd.read_csv(self.config['file_path'])
        self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        self.df = self.df.set_index('datetime').sort_index()

        # [보수적 수정] 결측치가 너무 많은 앞부분 제거
        self.df.dropna(inplace=True)
        print(f"    >>> 데이터 로드 완료: {self.df.shape}")

        # [수정] 레이블 존재 여부 확인
        if 'target_class' in self.df.columns and 'target_return' in self.df.columns:
            self.is_labeled = True
            print("    >>> [Notice] 'target_class'와 'target_return'이 이미 존재합니다. 레이블링 단계는 건너뜁니다.")

        # [수정] labeling이 필요할 경우, 핵심 가격 컬럼 존재 여부 확인
        elif not all(col in self.df.columns for col in ['close', 'high', 'low']):
            raise KeyError(
                "Triple Barrier 레이블링에 필수적인 'close', 'high', 'low' 컬럼이 데이터셋에 없습니다. "
                "레이블링을 건너뛰려면 'target_class'와 'target_return' 컬럼을 데이터셋에 포함하세요."
            )

    # =============================================================================
    # 2. Triple Barrier (24h 최적화)
    # =============================================================================
    def _get_volatility(self, prices, span=100):
        # 일간 수익률이 아닌, 캔들 간 수익률의 EWMA 변동성
        returns = prices.pct_change()
        return returns.ewm(span=span).std()

    def apply_labeling(self):
        # [수정] 이미 레이블이 있다면 건너뜀
        if self.is_labeled:
            print("[2] 레이블이 이미 존재하여 Triple Barrier 레이블링을 건너뜁니다.")
            return

        print("[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...")

        # 1) 변동성 계산 (기간: 1440분)
        self.df['volatility'] = self._get_volatility(
            self.df['close'],
            span=self.config['triple_barrier']['span']
        )

        t_final = self.config['triple_barrier']['vertical_barrier']
        pt = self.config['triple_barrier']['pt']
        sl = self.config['triple_barrier']['sl']

        fee_one_way = self.config['fee_rate']
        slip_one_way = self.config['slippage']
        round_trip_cost = 2 * (fee_one_way + slip_one_way)

        # [수정] 긴 호흡이므로 목표 수익률을 조금 더 여유 있게 잡음
        multiplier = 1.2
        extra_margin = 0.001    # 0.1% 추가 마진
        MIN_RET = round_trip_cost * multiplier + extra_margin

        print(f"    >>> Holding Period: {t_final} mins (24h)")
        print(f"    >>> MIN_RET (Target): {MIN_RET*100:.3f}%")

        labels = []
        returns = []
        closes = self.df['close'].values
        vols   = self.df['volatility'].values
        highs  = self.df['high'].values
        lows   = self.df['low'].values
        n_samples = len(closes)

        # 벡터 연산으로 최적화 가능하나, 로직 명확성을 위해 기존 Loop 유지 (보수적 접근)
        for i in range(n_samples - t_final):
            current_price = closes[i]
            vol = vols[i]
            limit_vol = max(vol, 0.002) # [수정] 24시간 변동성 최소값 보정 (너무 작으면 안됨)

            upper = current_price * (1 + limit_vol * pt)
            lower = current_price * (1 - limit_vol * sl)

            label = 0
            ret = 0.0
            touched = False

            # Horizon Loop
            for j in range(1, t_final + 1):
                # Take Profit
                if highs[i + j] >= upper:
                    ret = (upper - current_price) / current_price
                    if ret > MIN_RET: label = 1
                    touched = True
                    break
                # Stop Loss
                if lows[i + j] <= lower:
                    ret = (lower - current_price) / current_price
                    label = 0
                    touched = True
                    break

            # Time Limit (Vertical Barrier)
            if not touched:
                final_price = closes[i + t_final]
                ret = (final_price - current_price) / current_price
                if ret > MIN_RET: label = 1
                else: label = 0

            labels.append(label)
            returns.append(ret)

        # Padding for the end
        labels.extend([np.nan] * t_final)
        returns.extend([np.nan] * t_final)

        self.df['target_class'] = labels
        self.df['target_return'] = returns
        self.df.dropna(subset=['target_class'], inplace=True)

        pos_ratio = self.df['target_class'].mean() * 100
        print(f"    >>> 레이블링 완료. 상승(1) 비율: {pos_ratio:.2f}%")

    # =============================================================================
    # 3. 데이터 분할 및 정규화 (Multi-Window Scaling)
    # =============================================================================
    def split_and_scale(self):
        """
        [수정] 24시간 예측을 위해 장기 Window를 기준으로 Scaling 수행
        """
        print("[3] 데이터 분할 및 스케일링 (24h Optimized)...")
        # [수정] 가격 컬럼이 없을 수도 있으므로, 제외 목록에서 가격 컬럼 체크 로직 제거
        exclude = ['target_class', 'target_return', 'datetime', 'volatility']

        # DataFrame 컬럼 중 exclude에 없는 컬럼을 모두 feature로 사용
        # (open, high, low, close, volume 컬럼이 없다고 가정)
        base_features = [c for c in self.df.columns if c not in exclude]

        # [핵심 수정] 단일 200 윈도우 -> 1440(1일) 윈도우 사용
        # 이유: 24시간 뒤를 예측하려면, 현재 데이터가 지난 24시간 대비 어디에 위치하는지(Z-score)가 가장 중요함.
        main_window = self.config['feature_windows'][0] # 1440

        X_df = pd.DataFrame(index=self.df.index)

        # 기존 피처들을 24시간 기준 Rolling Z-Score로 변환
        # (값이 클수록 지난 24시간 평균 대비 고평가, 작으면 저평가)
        for col in base_features:
            # 롤링 윈도우 계산 시 최소 윈도우 크기를 설정 (데이터 안정성 확보)
            roll_mean = self.df[col].rolling(window=main_window, min_periods=int(main_window * 0.8)).mean()
            roll_std = self.df[col].rolling(window=main_window, min_periods=int(main_window * 0.8)).std().replace(0, 1)

            # Z-Score Normalization
            X_df[col] = (self.df[col] - roll_mean) / roll_std

        # NaN 제거 (1440분 데이터 확보 필요)
        X_df.dropna(inplace=True)

        # 인덱스 동기화
        common_idx = X_df.index.intersection(self.df.index)
        X_scaled = X_df.loc[common_idx]

        # Train/Test Split
        split_idx = int(len(X_scaled) * (1 - self.config['test_size']))

        self.X_train = X_scaled.iloc[:split_idx].clip(-5, 5)
        self.X_test = X_scaled.iloc[split_idx:].clip(-5, 5)

        self.y_class_train = self.df.loc[X_scaled.index, 'target_class'].iloc[:split_idx]
        self.y_class_test = self.df.loc[X_scaled.index, 'target_class'].iloc[split_idx:]
        self.y_return_test = self.df.loc[X_scaled.index, 'target_return'].iloc[split_idx:]

        print(f"    >>> 학습 셋: {self.X_train.shape} (Window: {main_window})")

    # =============================================================================
    # 4. Purged K-Fold CV
    # =============================================================================
    def _purged_cv_score(self, model, X, y):
        kf = KFold(n_splits=3, shuffle=False)
        scores = []
        gap = self.config['purge_gap']  # 1500 (24h 이상)

        for tr_idx, val_idx in kf.split(X):
            # [중요] Validation 시작점보다 'gap'만큼 이전에 끝나는 데이터만 학습에 사용
            # 24시간 보유 전략이므로, 최소 24시간 전 데이터까지만 봐야 함.
            tr_idx = tr_idx[tr_idx < val_idx[0] - gap]

            if len(tr_idx) < 1000: continue # 데이터 부족 시 스킵

            model.fit(X.iloc[tr_idx], y.iloc[tr_idx], verbose=False)

            # [수정] CV 스코어 계산 시에도 Margin -> Proba 변환 명시
            # Scikit-Learn Wrapper는 Custom Objective 사용 시 predict()가 Margin을 반환할 수 있음
            try:
                # 안전한 방법: Margin을 얻어서 Sigmoid
                pred_margin = model.predict(X.iloc[val_idx], output_margin=True)
                pred_proba = 1.0 / (1.0 + np.exp(-pred_margin))
                pred_label = (pred_proba > 0.5).astype(int)
            except TypeError:
                # predict_proba가 정상 작동하는 경우
                pred_label = model.predict(X.iloc[val_idx])

            scores.append(accuracy_score(y.iloc[val_idx], pred_label))

        return np.mean(scores) if scores else 0

    # =============================================================================
    # 5. Optuna 튜닝
    # =============================================================================
    def run_optuna(self):
        print(f"[4] 하이퍼파라미터 튜닝 (Trials: {self.config['optuna_trials']})")

        def objective_cls(trial):
            fl_alpha = trial.suggest_float('fl_alpha', 0.85, 0.95) # 범위 소폭 확대
            fl_gamma = trial.suggest_float('fl_gamma', 0.5, 2.0)
            fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)

            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 2000), # 긴 호흡 데이터라 복잡도 증가 예상
                'max_depth': trial.suggest_int('max_depth', 4, 15),
                'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 0.1, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
                'gamma': trial.suggest_float('xgb_gamma', 0.0, 5.0),

                'objective': fl_obj.get_objective,
                'eval_metric': 'logloss',
                'disable_default_eval_metric': 1,
                'tree_method': 'hist',
                'device': 'cuda',
                'n_jobs': -1,
                'random_state': self.config['random_state']
            }

            model = xgb.XGBClassifier(**params)
            return self._purged_cv_score(model, self.X_train, self.y_class_train)

        study_cls = optuna.create_study(direction='maximize')
        study_cls.optimize(objective_cls, n_trials=self.config['optuna_trials'])

        self.best_params_class = study_cls.best_params
        print(f"    >>> Best Params: {self.best_params_class}")
        return study_cls

    # =============================================================================
    # 6. 최종 모델 학습
    # =============================================================================
    def train_final_models(self):
        print("[5] 최종 모델 학습...")
        bp = self.best_params_class.copy()
        fl_alpha = bp.pop('fl_alpha')
        fl_gamma = bp.pop('fl_gamma')

        if 'xgb_gamma' in bp: bp['gamma'] = bp.pop('xgb_gamma')

        fl_obj = FocalLossObjective(alpha=fl_alpha, gamma=fl_gamma)
        self.classifier = xgb.XGBClassifier(
            **bp,
            objective=fl_obj.get_objective,
            scale_pos_weight=1.0,
            n_jobs=-1
        )
        self.classifier.fit(self.X_train, self.y_class_train)
        print(f"    >>> 모델 학습 완료.")
        return self.classifier

    def _find_optimal_threshold(self, pred_probs, y_true):
        precision, recall, thresholds = precision_recall_curve(y_true, pred_probs)
        numerator = 2 * precision * recall
        denominator = precision + recall
        f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)
        best_idx = np.argmax(f1_scores)
        optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        max_f1 = f1_scores[best_idx]
        return optimal_threshold, max_f1
    def evaluate(self):
        # 1. 모델 예측
        pred_probs = self.classifier.predict_proba(self.X_test)[:, 1]
        probs_series = pd.Series(pred_probs)

        print("\n" + "="*60)
        print(" [ 24시간 스윙 전략 상세 평가 (Enhanced Metrics) ]")
        print("="*60)

        # ---------------------------------------------------------
        # [Diagnosis] 예측 확률 분포 진단
        # ---------------------------------------------------------
        print("\n[1] 예측 확률 분포 (Confidence Distribution):")
        print(probs_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

        # 비용 설정
        fee = self.config['fee_rate']
        slip = self.config['slippage']
        cost = 2 * (fee + slip)

        # ---------------------------------------------------------
        # [Threshold] 최적화 및 적용
        # ---------------------------------------------------------
        opt_th, max_f1 = self._find_optimal_threshold(pred_probs, self.y_class_test)

        # [수정] 0.5가 너무 낮아 손실이 발생하는 경우를 방지하기 위해
        # F1 최적값과 0.55(약간의 우위) 중 큰 값을 선택하도록 보수적 설정
        final_th = max(opt_th, 0.55)

        # 안전장치: 상위 30% 이내 확률일 때만 진입 (노이즈 필터링)
        safety_th = probs_series.quantile(0.70)
        if final_th < safety_th:
            final_th = safety_th

        print(f"\n[2] 임계값 설정 (Threshold):")
        print(f"    >>> F1 Max Threshold: {opt_th:.4f} (Score: {max_f1:.4f})")
        print(f"    >>> Safety Threshold (Top 30%): {safety_th:.4f}")
        print(f"    >>> 최종 적용 Threshold: {final_th:.4f}")

        # ---------------------------------------------------------
        # [Backtest] 시뮬레이션
        # ---------------------------------------------------------
        returns_arr = self.y_return_test.values
        executed_trades = []

        # [정책] 단일 포지션 사이클 준수: 진입 후 보유기간(24h) 동안 재진입 금지
        # 5분 쿨다운은 다중 포지션을 의미하므로 자금 관리상 위험할 수 있어
        # 보수적으로 vertical_barrier(1440분)를 쿨다운으로 사용합니다.
        cooldown = self.config['triple_barrier']['vertical_barrier']
        next_trade_idx = 0

        for i in range(len(pred_probs)):
            if i < next_trade_idx: continue

            if pred_probs[i] >= final_th:
                # 수익률 계산 (비용 차감)
                raw_return = returns_arr[i]
                net_return = raw_return - cost
                executed_trades.append(net_return)

                # 포지션 점유 처리
                next_trade_idx = i + cooldown

        if not executed_trades:
            print("\n>>> [Warning] 조건에 맞는 진입 신호가 없습니다. (No Trades)")
            return

        executed_trades = np.array(executed_trades)
        n_trades = len(executed_trades)

        # ---------------------------------------------------------
        # [Statistics] 상세 성과 지표 계산
        # ---------------------------------------------------------
        # 1. 승률
        win_trades = executed_trades[executed_trades > 0]
        loss_trades = executed_trades[executed_trades <= 0]
        win_rate = len(win_trades) / n_trades * 100

        # 2. 수익률
        avg_ret = np.mean(executed_trades) * 100
        cum_ret = (np.prod(executed_trades + 1) - 1) * 100
        std_ret = np.std(executed_trades)

        # 3. MDD (Maximum Drawdown)
        cum_returns_series = np.cumprod(1 + executed_trades)
        peak = np.maximum.accumulate(cum_returns_series)
        drawdown = (cum_returns_series - peak) / peak
        max_mdd = drawdown.min() * 100

        # 4. Sharpe Ratio (무위험 수익률 0 가정)
        # 연율화: 데이터 기간에 따라 다르지만 여기서는 단순히 거래 횟수 기반 제곱근 사용
        sharpe = (np.mean(executed_trades) / std_ret) * np.sqrt(n_trades) if std_ret > 0 else 0

        # 5. 손익비 (Profit Factor)
        avg_win = win_trades.mean() if len(win_trades) > 0 else 0
        avg_loss = loss_trades.mean() if len(loss_trades) > 0 else 0
        pnl_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else 0

        print("-" * 50)
        print(f" [최종 성과 보고서 (N={n_trades})]")
        print("-" * 50)
        print(f" 1. 승률 (Win Rate)       : {win_rate:.2f}%")
        print(f" 2. 평균 수익률 (Avg Ret) : {avg_ret:.4f}%")
        print(f" 3. 누적 수익률 (Cum Ret) : {cum_ret:.2f}%")
        print(f" 4. 샤프 지수 (Sharpe)    : {sharpe:.4f}")
        print(f" 5. MDD (Max Drawdown)    : {max_mdd:.2f}%")
        print(f" 6. 손익비 (P/L Ratio)    : {pnl_ratio:.2f}")
        print("-" * 50)

if __name__ == "__main__":
    bot = BitcoinTradingModel(CONFIG)
    bot.load_data()

    # [수정] 레이블이 이미 존재하면 apply_labeling 건너뛰기
    if not bot.is_labeled:
        bot.apply_labeling()

    bot.split_and_scale()
    bot.run_optuna()
    model = bot.train_final_models()
    bot.evaluate()

[1] 데이터 로드 중... (/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv)
    >>> 데이터 로드 완료: (475425, 128)
[2] Triple Barrier 레이블링 (Cost-aware, 24h Horizon) 적용 중...
    >>> Holding Period: 1440 mins (24h)
    >>> MIN_RET (Target): 0.208%
    >>> 레이블링 완료. 상승(1) 비율: 19.20%
[3] 데이터 분할 및 스케일링 (24h Optimized)...


[I 2025-12-03 01:03:52,799] A new study created in memory with name: no-name-839963ef-62fd-4696-bc69-97c029538b8c


    >>> 학습 셋: (378267, 127) (Window: 1440)
[4] 하이퍼파라미터 튜닝 (Trials: 100)


[I 2025-12-03 01:04:37,124] Trial 0 finished with value: 0.7103553838955023 and parameters: {'fl_alpha': 0.8547733723783051, 'fl_gamma': 1.6372764264732942, 'n_estimators': 1028, 'max_depth': 14, 'learning_rate': 0.02270373218863236, 'subsample': 0.8215914937248029, 'colsample_bytree': 0.6956662662708948, 'reg_alpha': 0.007011990515979247, 'reg_lambda': 0.0072962757404493855, 'xgb_gamma': 2.1475707130880624}. Best is trial 0 with value: 0.7103553838955023.
[I 2025-12-03 01:05:46,029] Trial 1 finished with value: 0.6850399321114451 and parameters: {'fl_alpha': 0.8542905597139523, 'fl_gamma': 0.7061074739875879, 'n_estimators': 1903, 'max_depth': 8, 'learning_rate': 0.012011517290555225, 'subsample': 0.945155383458852, 'colsample_bytree': 0.8509829795379056, 'reg_alpha': 0.03434422288220108, 'reg_lambda': 0.04997947714436586, 'xgb_gamma': 3.9258551150237784}. Best is trial 0 with value: 0.7103553838955023.
[I 2025-12-03 01:06:26,704] Trial 2 finished with value: 0.7387281999222771 and pa

    >>> Best Params: {'fl_alpha': 0.8521428597341948, 'fl_gamma': 0.5006196588732799, 'n_estimators': 1699, 'max_depth': 15, 'learning_rate': 0.004901906023463469, 'subsample': 0.6489989988159443, 'colsample_bytree': 0.8924419136508435, 'reg_alpha': 0.002833588306801283, 'reg_lambda': 0.06089027424847081, 'xgb_gamma': 0.11785953484198566}
[5] 최종 모델 학습...
    >>> 모델 학습 완료.

 [ 24시간 스윙 전략 상세 평가 (Enhanced Metrics) ]

[1] 예측 확률 분포 (Confidence Distribution):
count    94567.000000
mean         0.218146
std          0.155977
min          0.008139
50%          0.176137
75%          0.301264
90%          0.444940
95%          0.534831
99%          0.697555
max          0.881875
dtype: float64

[2] 임계값 설정 (Threshold):
    >>> F1 Max Threshold: 0.0180 (Score: 0.3336)
    >>> Safety Threshold (Top 30%): 0.2699
    >>> 최종 적용 Threshold: 0.5500
--------------------------------------------------
 [최종 성과 보고서 (N=52)]
--------------------------------------------------
 1. 승률 (Win Rate)       : 42.31%
 2.

In [3]:
save_path = "/content/drive/MyDrive/btc_trading_bot/xgb_final_model.joblib"

joblib.dump(bot.classifier, save_path)
print("saved to:", save_path)

saved to: /content/drive/MyDrive/btc_trading_bot/xgb_final_model.joblib
